## Coma Galaxy List Create

During the analysis of the Coma galaxies, we need a list of the 'main' galaxies with details, including position (ra/dec), magnitude, and effective radius. 

This information we can get from a number of sources, including details from the original Trentham spreadsheet, but augmented with up to date information from (preferrentially) Simbad or (fallback) NED.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.coordinates import SkyCoord
import astropy.units as u
import logging


In [2]:

def setup_logging(verbose=False):
    """
    Call this once at the top of the notebook.
    If verbose=True, DEBUG messages will show; otherwise only INFO+.
    """
    name = "Coma_GCs_debug"
    
    # Remove any root handlers (if I had root-level logging before)
    for h in logging.root.handlers[:]:
        logging.root.removeHandler(h)

    # Get named logger and clear *its* handlers + disable propagation
    logger = logging.getLogger(name)
    for h in logger.handlers[:]:
        logger.removeHandler(h)
    logger.propagate = False

    # Set logger’s level
    logger.setLevel(logging.DEBUG)   # we capture everything here

    # INFO-only handler (no timestamp)
    info_handler = logging.StreamHandler()
    info_handler.setLevel(logging.INFO)
    info_handler.addFilter(lambda rec: rec.levelno == logging.INFO)
    info_handler.setFormatter(logging.Formatter("%(message)s"))
    logger.addHandler(info_handler)

    # non-INFO handler (timestamped, DEBUG or WARNING+)
    other = logging.StreamHandler()
    other.setLevel(logging.DEBUG if verbose else logging.WARNING)
    other.addFilter(lambda rec: rec.levelno != logging.INFO)
    # allow only your logger’s DEBUG (if you still want a name-filter):
    other.addFilter(lambda rec: rec.levelno != logging.DEBUG or rec.name == name)
    fmt = "[%(levelname)s] %(asctime)s.%(msecs)03d %(message)s"
    other.setFormatter(logging.Formatter(fmt, datefmt="%H:%M:%S"))
    logger.addHandler(other)


In [4]:

# Set verbosity here:
setup_logging(verbose=True)   # or False

# testing logging
logger = logging.getLogger("Coma_GCs_debug")
logger.debug("Debug messages are ON")
logger.info("Info messages are always shown")
logger.warning("Warnings also show up")


[DEBUG] 16:06:49.603 Debug messages are ON
Info messages are always shown
[WARNING] 16:06:49.605 Warnings also show up


In [ ]:
# Define our list of galaxies of interest, including the BCGs and IC 4051 which we load from a csv file

gals_df = pd.read_csv('data/Notable_Gals+counts.csv', 
                      names=['name', 'mv', 'velocity', 'name2', 'z', 'Dist_Mpc', 'Sigma_GC', 'MV', 'Sn', 'Ncss', 
                             'N_GC_Obs_8Re', 'gcs_uncert', 'N_UCD_Obs_8Re', 'ucd_uncert', 
                             'N_CSS_Obs_8Re', 'CSS_uncert', 'Frac', 'Label1', 'Label2',
                             'Sn_obs', 'Sn_obs_uncert', 'Sn_obs_adj', 'Sn_obs_uncert_adj'], 
                      index_col=False, header=0)

gals_df.head(5)



In [ ]:
# Set up values for Re, used previously.

notable_gal_2018 = {'galname': ['IC 3973', 'IC 3976', 'IC 3998', 'IC 4011', 'IC 4012', 'IC 4021', 
                                'IC 4026', 'IC 4030', 'IC 4033', 'IC 4040', 'IC 4041', 'IC 4045', 
                                'IC 4051', 'NGC 4867', 'NGC 4869', 'NGC 4871', 'NGC 4872', 'NGC 4873',
                                'NGC 4874', 'NGC 4875', 'NGC 4876','NGC 4882', 'NGC 4883', 'NGC 4889', 
                                'NGC 4894', 'NGC 4898', 'NGC 4906', 'NGC 4908', 
                                'LEDA 44651', 'LEDA 44636', 'LEDA 44656', 'LEDA 44652', 'LEDA 44635', 'IC 3968', 'WISEA J125925.33+275804.6', 'Z 160-233', 'LEDA 44594', 'LEDA 126792', 'LEDA 44644', '2MASX J12595013+2755292', '2MASX J12592136+2758248', 'LEDA 1821898', '2MASX J12592136+2758248', '2MASX J12592536+2756038', 'LEDA 1821555', '2MASX J12592265+2753488', '2MASX J12595013+2755292', 'LEDA 1821341', 'SDSS J125948.57+275857.7', 'ECO 11254', 'LEDA 126781', 'SDSS J125946.71+280000.4', 'LEDA 126785', 'LEDA 126788', 'SDSS J125930.25+280115.0', 'SDSS J125928.50+280109.3', 'CAIRNS J125935.95+275421.4', 'SDSS J125923.41+275510.4', '[ARF2020] COMA 22 0296',  # Line for NGC 4874 satellites
                                'LEDA 93586', 'LEDA 44708', 'LEDA 44707', '2MASX J13001036+2757332', 'LEDA 126771', 'LEDA 126768', 'SDSS J130008.39+275716.7', 'LEDA 44693', 'LEDA 44692', '2MASX J12595670+2755483', 'CAIRNS J130005.15+275835.8', '2MASS J13001763+2759145', 'LEDA 126759', 'LEDA 3098454', 'SDSS J130017.67+275718.9', # Line for NGC 4889 satellites
                                'LEDA 44792', 'IC 4042', 'LEDA 44809', 'SDSS J130041.19+280242.4', 'LEDA 44815', 'SDSS J130051.15+280249.7', 'LEDA 44863', 'GMP 2297', 'GMP 2311', '2MASX J13005468+2759512', '[EDG2007] 489', 'GMP 2399',  # Line for IC 4051 satellites
                                '2MASX J13001702+2803502', 'LEDA 44723', 'SDSS J130018.34+280333.4', 'LEDA 126763', 'LEDA 126764', 'SDSS J130026.15+280032.0', 'SDSS J130025.97+280344.6', 'SDSS J130027.57+280323.9', 'LEDA 126761', 'LEDA 126754',  # line for IC 4026 and surrounding
                                'SDSS J130033.33+275849.3', 'LEDA 1822111', 'LEDA 1821892', 'LEDA 126756', 'LEDA 126758', 'SDSS J130022.65+275754.8', 'SDSS J130036.67+275427.4', 'SDSS J130032.48+275833.2', # line for IC 4030 / IC 4033
                                'LEDA 44616', 'LEDA 44602', 'NGC 4864', 'IC 3955', '2MASX J12592016+2753098', '2MASX J12590459+2754389', '2MASX J12594422+2752037', '2MASX J12594610+2751257', 'LEDA 44654', '2MASX J12593697+2749327', 'LEDA 2816216', 'LEDA 126789', 'IC 3960', 'LEDA 126801', 'LEDA 126800', 'LEDA 93697', 'LEDA 93696', 'LEDA 1822852', 'LEDA 1822726', '2MASX J12591389+2804349', 'NGC 4865', '2MASX J12592022+2804278', 'LEDA 44609', # NGC 4874 outskirts (2-5, 2-6, 2-7, 3-6, 4-6, 5-5)
                                'CAIRNS J125947.21+280315.7', '[EDG2007] 205', '2MASX J12595140+2804232', '2MASX J12595760+2803543', 'GMP 3016', '2MASS J13001968+2807172', 'SDSS J130042.56+280658.6', # North side
                                'LEDA 126747', 'GMP 2267', '2MASX J13004737+2755196', '2MASX J13005921+2753592', 'LEDA 1820946', 'GMP 2273', 'GMP 2287'  # east side
                               ],
                    'Mv_deVac_1991': [14.37, 14.70, 14.57, 15.12, 14.94, 14.85, 
                                      14.59, 15.40, 15.21, 14.84, 14.35, 13.94, 
                                      13.63, 14.46, 13.76, 14.14, 14.41, 14.11,
                                      11.68, 14.65, 14.39, 13.86, 14.35, 11.49, 
                                      15.19, 13.48, 14.10, 13.18, 
                                      16.02, 15.89, 15.15, 15.65, 16.60, 15.32, 15.59, 15.42, 15.87, 17.17, 15.59, 16.05, 16.12, 18.055, 16.12, 17.02, 17.59, 15.33, 16.05, 17.4, 17.69, 17.445, 17.43, 17.8, 18.38, 18.62, 17.82, 18.07, 18.345, 17.692, 17.13,
                                      15.44, 15.99, 15.42, 17.406, 17.26, 17.58, 18.03, 16.92, 17.0, 15.32, 17.075, 17.86, 17.36, 14.33, 13.862,
                                      14.844, 14.01, 14.96, 17.15, 15.59, 17.077, 16.34, 18.92, 18.73, 19.06, 19.644, 17.63,
                                      15.52, 15.17, 17.34, 17.31, 16.72, 17.57, 18.05, 19.31, 17.04, 17.15,
                                      17.93, 16.79, 16.81, 16.57, 16.81, 18.3, 17.87, 18.52,
                                      15.014, 16.033, 13.872, 14.31, 15.38, 15.46, 16.24, 15.0, 16.34, 16.43, 16.37, 17.13, 14.47, 17.18, 16.76, 16.31, 16.53, 17.13, 17.7, 14.89, 13.692, 15.94, 15.56,
                                      18.03, 16.402, 16.23, 16.59, 17.69, 17.74, 18.24,
                                      17.24, 18.29, 15.47, 14.61, 16.88, 18.63, 18.73
                                     ],
                    'Re_arcsec': [2.613105, 3.80061, 8.921385, 5.258385, 2.77002, 5.341545,
                                  7.354215, 5.68805, 8.965935, 7.5, 7.68141, 4.39461,     # IC 4040 was 0.270000
                                  8.200000, 2.84000, 5.240000, 7.471035, 2.71000, 9.142155,
                                  19.40000, 2.73000, 6.080085, 4.50000, 4.740000, 15.30000, 
                                  4.91436, 4.340655, 8.006130, 10.71000, 
                                  None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None,
                                  None, None, None, None, None, None, None, None, None, None, None, None, None, None, None,
                                  None, None, None, None, None, None, None, None, None, None, None, None,
                                  None, None, None, None, None, None, None, None, None, None, 
                                  None, None, None, None, None, None, None, None,
                                  None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None,
                                  None, None, None, None, None, None, None,
                                  None, None, None, None, None, None, None
                                 ],
                    'show_label': [True, True, True, True, True, True,
                                   True, True, True, True, True, True,
                                   True, True, True, True, True, True,
                                   True, True, True, True, True, True,
                                   True, True, True, True, 
                                   False, False, False, False, False, True,  False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, 
                                   False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, 
                                   False, True,  False, False, False, False, False, False, False, False, False, False,
                                   False, False, False, False, False, False, False, False, False, False,
                                   False, False, False, False, False, False, False, False,
                                   False, False, True,  True,  False, False, False, False, False, False, False, False, True,  False, False, False, False, False, False, False, True,  False, False,
                                   False, False, False, False, False, False, False,
                                   False, False, False, False, False, False, False
                                  ]
                   }
# NGC 4872, NGC 4875, NGC 4882, NGC 4908 added Re from NED (SDSS) 2007SDSS6.C...0000: http://www.sdss.org/dr6/products/catalogs/index.html
notable_gal_2018_df = pd.DataFrame(notable_gal_2018)

notable_gal_2018_df.head(5)


In [ ]:
def read_simbad_dimensions(tbl):
    out = {}
    for name, default_u in [
        ("galdim_majaxis", u.arcmin),
        ("galdim_minaxis", u.arcmin),
        ("galdim_angle",   u.deg),
    ]:
        if name in tbl.colnames:
            col = tbl[name]
            # handle masked / missing values
            val = None if (hasattr(col, "mask") and bool(col.mask[0])) else col[0]
            unit = col.unit or col.info.unit or default_u
            out[name] = None if val is None else (val * unit)
        else:
            out[name] = None
    return out
    

In [ ]:
from astroquery.simbad import Simbad
from astroquery.ipac.ned import Ned
from astropy import units as u
from astropy.coordinates import SkyCoord
from astropy.cosmology import Planck15 as cosmo

# tell Simbad to return redshifts
Simbad.add_votable_fields('rvz_redshift')
Simbad.add_votable_fields('V', 'dimensions')  # 'V' mag, and dim_majaxis/minaxis/angle

galaxy_list = [
    "IC 3973", "IC 3976", "IC 3998", "IC 4011", "IC 4012", "IC 4021", "IC 4026",
    "IC 4030", "IC 4033", "IC 4040", "IC 4041", "IC 4045", "IC 4051",
    "NGC 4867", "NGC 4869", "NGC 4871", "NGC 4872", "NGC 4873", "NGC 4874",
    "NGC 4875", "NGC 4876", "NGC 4882", "NGC 4883", "NGC 4889", "NGC 4894",
    "NGC 4898", "NGC 4906", "NGC 4908", "LEDA 44651", 'LEDA 44636', 'LEDA 44656', 'LEDA 44652', 'LEDA 44635', 'IC 3968', 'WISEA J125925.33+275804.6', 'Z 160-233', 'LEDA 44594', 'LEDA 126792', 'LEDA 44644', '2MASX J12595013+2755292', '2MASX J12592136+2758248', 'LEDA 1821898', '2MASX J12592136+2758248', '2MASX J12592536+2756038', 'LEDA 1821555', '2MASX J12592265+2753488', '2MASX J12595013+2755292', 'LEDA 1821341', 'SDSS J125948.57+275857.7', 'ECO 11254', 'LEDA 126781', 'SDSS J125946.71+280000.4', 'LEDA 126785', 'LEDA 126788', 'SDSS J125930.25+280115.0', 'SDSS J125928.50+280109.3', 'CAIRNS J125935.95+275421.4', 'SDSS J125923.41+275510.4', '[ARF2020] COMA 22 0296',
    'LEDA 93586', 'LEDA 44708', 'LEDA 44707', '2MASX J13001036+2757332', 'LEDA 126771', 'LEDA 126768', 'SDSS J130008.39+275716.7', 'LEDA 44693', 'LEDA 44692', '2MASX J12595670+2755483', 'CAIRNS J130005.15+275835.8', '2MASS J13001763+2759145', 'LEDA 126759', 'LEDA 3098454', 'SDSS J130017.67+275718.9',
    'LEDA 44792', 'IC 4042', 'LEDA 44809', 'SDSS J130041.19+280242.4', 'LEDA 44815', 'SDSS J130051.15+280249.7', 'LEDA 44863', 'GMP 2297', 'GMP 2311', '2MASX J13005468+2759512', '[EDG2007] 489', 'GMP 2399',
    '2MASX J13001702+2803502', 'LEDA 44723', 'SDSS J130018.34+280333.4', 'LEDA 126763', 'LEDA 126764', 'SDSS J130026.15+280032.0', 'SDSS J130025.97+280344.6', 'SDSS J130027.57+280323.9', 'LEDA 126761', 'LEDA 126754',
    'SDSS J130033.33+275849.3', 'LEDA 1822111', 'LEDA 1821892', 'LEDA 126756', 'LEDA 126758', 'SDSS J130022.65+275754.8', 'SDSS J130036.67+275427.4', 'SDSS J130032.48+275833.2',
    'LEDA 44616', 'LEDA 44602', 'NGC 4864', 'IC 3955', '2MASX J12592016+2753098', '2MASX J12590459+2754389', '2MASX J12594422+2752037', '2MASX J12594610+2751257', 'LEDA 44654', '2MASX J12593697+2749327', 'LEDA 2816216', 'LEDA 126789', 'IC 3960', 'LEDA 126801', 'LEDA 126800', 'LEDA 93697', 'LEDA 93696', 'LEDA 1822852', 'LEDA 1822726', '2MASX J12591389+2804349', 'NGC 4865', '2MASX J12592022+2804278', 'LEDA 44609',
    'CAIRNS J125947.21+280315.7', '[EDG2007] 205', '2MASX J12595140+2804232', '2MASX J12595760+2803543', 'GMP 3016', '2MASS J13001968+2807172', 'SDSS J130042.56+280658.6',
    'LEDA 126747', 'GMP 2267', '2MASX J13004737+2755196', '2MASX J13005921+2753592', 'LEDA 1820946' 
]

results = []

for galaxy in galaxy_list:
    logger.info(f'\nQuerying Simbad for galaxy {galaxy}')
    ra_deg = None
    dec_deg = None
    redshift = None
    distance_mpc = None
    dist_method = None    # no analog in Simbad
    src_arch = None
    Vmag = None
    maj_arcmin = None
    min_arcmin = None
    pa_deg = None
    
    # --- 1) query Simbad for coords + redshift ---
    try:
        tbl = Simbad.query_object(galaxy)
        if tbl is not None and len(tbl) > 0:  # query Simbad first
            dims = read_simbad_dimensions(tbl)

            ra_str  = tbl['ra'][0]        # e.g. '12 30 49.42338'
            dec_str = tbl['dec'][0]       # e.g. '+12 23 28.0439'
            redshift = tbl['rvz_redshift'][0]  # may be masked if unknown
            logger.debug(f'Simbad galaxy coords are RA:{tbl['ra'][0]:.5f}, Dec:{tbl['dec'][0]:.5f}, redshift: {tbl['rvz_redshift'][0]:.4f}')

            # parse to a SkyCoord and extract decimal degrees
            coord = SkyCoord(f"{ra_str} {dec_str}", unit=(u.deg, u.deg))
            ra_deg  = coord.ra.deg
            dec_deg = coord.dec.deg
            src_arch = 'Simbad'

            # V magnitude (may be masked/NaN if unknown)
            if 'V' in tbl.colnames:
                Vval = tbl['V'][0]
                Vmag = float(Vval) if Vval is not None and Vval == Vval else None  # NaN-safe
            # Sizes (major/minor axes are the “dimensions” group)
            for name in ('dim_majaxis', 'dim_minaxis', 'dim_angle'):
                if name not in tbl.colnames:
                    continue
            maj_arcmin = float(tbl['dim_majaxis'][0]) if 'dim_majaxis' in tbl.colnames else None
            # logger.debug(f'Major axis dim: {tbl['dim_majaxis'].unit}')
            min_arcmin = float(tbl['dim_minaxis'][0]) if 'dim_minaxis' in tbl.colnames else None
            # logger.debug(f'Minor axis dim: {tbl['dim_minaxis'].unit}')
            pa_deg     = float(tbl['dim_angle'][0])   if 'dim_angle'   in tbl.colnames else None

        else: 
            logger.warning(f'Falling back to NED for galaxy {galaxy}')
            obj = Ned.query_object(galaxy)  # otherwise 
            logger.debug(f'NED galaxy coords are RA:{obj['RA'].data[0]:.5f}, Dec:{obj['DEC'].data[0]:.5f}, redshift: {obj['Redshift'][0]:.4f}')
            redshift = obj['Redshift'][0]
            ra_deg = obj['RA'].data[0]
            dec_deg = obj['DEC'].data[0]
            src_arch = 'NED'

    except Exception as e:
        logger.debug(f'Exception {e} raised')
        pass

    # --- 2) compute a distance from the redshift (if available) ---
    if redshift is not None and redshift > 0:
        # this gives the luminosity distance; you can choose comoving_distance, angular_diameter_distance, etc.
        distance_mpc = cosmo.luminosity_distance(redshift).to(u.Mpc).value

    # --- 3) If SIMBAD didn’t have V or size, try NED tables ---
    if (Vmag is None) or (maj_arcmin is None):
        try:
            # Photometry: look for Johnson V (or any bandpass containing ' V ')
            phot = Ned.get_table(galaxy, table='photometry')  # photometry/diameters are supported tables
            # Columns vary a bit across objects; handle both common names:
            if Vmag is None and phot is not None and len(phot) > 0:
                band_col = 'NED Bandpass' if 'NED Bandpass' in phot.colnames else (
                           'Photometry Filter' if 'Photometry Filter' in phot.colnames else None)
                mag_col  = 'Magnitude' if 'Magnitude' in phot.colnames else None
                err_col  = 'Uncertainty' if 'Uncertainty' in phot.colnames else (
                           'Mag Error' if 'Mag Error' in phot.colnames else None)
                if band_col and mag_col:
                    # prefer a Johnson V if present, else the first row that clearly looks like V
                    v_rows = [row for row in phot if 'V' == str(row[band_col]).strip()
                              or ' Johnson V' in str(row[band_col])
                              or str(row[band_col]).strip() == 'V Johnson']
                    if not v_rows:
                        v_rows = [row for row in phot if 'V' in str(row[band_col])]
                    if v_rows:
                        # if errors exist, pick the smallest-uncertainty measurement
                        if err_col and all((row[err_col] is not None) for row in v_rows):
                            v_rows.sort(key=lambda r: (r[err_col] if r[err_col] is not None else float('inf')))
                        Vmag = float(v_rows[0][mag_col])

            # Diameters: use NED’s ‘diameters’ table (major/minor in arcmin, PA in deg)
            if maj_arcmin is None:
                diam = Ned.get_table(galaxy, table='diameters')
                if diam is not None and len(diam) > 0:
                    # Try to find columns regardless of exact wording/casing on NED’s side
                    def pick(col_hint):
                        for c in diam.colnames:
                            if col_hint.lower() in c.lower():
                                return c
                        return None
                    maj_col = pick('Major Axis')
                    min_col = pick('Minor Axis')
                    pa_col  = pick('Position Angle') or pick('PA')
                    if maj_col:
                        maj_arcmin = float(diam[maj_col][0])
                    if min_col:
                        min_arcmin = float(diam[min_col][0])
                    if pa_col:
                        pa_deg = float(diam[pa_col][0])
        except Exception as e:
            logger.debug(f'NED fallback failed for {galaxy}: {e}')

    results.append({
        'Galaxy': galaxy,
        'ra': ra_deg,
        'dec': dec_deg,
        'Redshift': redshift,
        'Distance (Mpc)': distance_mpc,
        'Distance Method': dist_method,
        'Source Archive': src_arch,
        'V (mag)': Vmag,
        'MajAxis (arcsec)': maj_arcmin,  # actually must be arcsec
        'MinAxis (arcsec)': min_arcmin,
        'PA (deg)': pa_deg,
    })

# now `results` has the same structure as before,
# except that distances are computed via cosmology.


In [ ]:
gals_posn = pd.DataFrame(results)
print(gals_posn)


In [5]:
# and then add distance calc from redshift
# NOTE: value of H0 used.

# import numpy as np

# H0 = 70  # km/s/Mpc
# c = 299792.458  # km/s

# gals_posn['Distance (Mpc)'] = c * gals_posn['Redshift'] / H0

# print(gals_posn)

In [ ]:
gals_df = pd.merge(notable_gal_2018_df, gals_df,  left_on='galname', right_on='name', how='outer')
print(len(gals_df))
gals_df.head(5)


In [ ]:
gals_df = pd.merge(gals_df, gals_posn, left_on='galname', right_on='Galaxy', how='outer')
print(len(gals_df))
gals_df.tail(5)


In [ ]:
gals_df.drop(columns=['name2','Galaxy', 'Distance (Mpc)', 'Distance Method', 'name', 'mv'], inplace=True)
gals_df = gals_df.rename(columns={'galname': 'name', 'Mv_deVac_1991': 'mv'})
gals_df['MV'] = gals_df['mv'] - 35
gals_df.tail(5)


In [ ]:
gals_df.columns



In [ ]:
gals_df[gals_df['Redshift']==0.044067]

In [ ]:
# gals_df[gals_df['Re_arcsec'] != gals_df['Re_arcsec']]['Re_arcsec']=gals_df['MajAxis (arcsec)']

# If no Re_arcsec value included, then use major axis from SimBad / NED

# gals_df['Re_arcsec'] = gals_df['Re_arcsec'].fillna(gals_df['MajAxis (arcsec)'])

mask = gals_df['Re_arcsec'].isna() & gals_df['MajAxis (arcsec)'].notna() & (gals_df['MajAxis (arcsec)'] >= 2.9)
gals_df.loc[mask, 'Re_arcsec'] = gals_df.loc[mask, 'MajAxis (arcsec)']


In [ ]:
gals_df.tail(5)

and finally save to a `csv` file so this can be used in the plotting routine (`Coma_plotting.ipynb`)

In [ ]:
gals_df.to_csv('data/gals_data_cleaned.csv', index=False)


## Attempt to obtain full galaxy dataset from archives

Although a manual dataset has been determined from the starting point of the 'notable' galaxies in the Madrid 2018 paper, we have criteria now which should allow the extraction of galaxies from the Simbad DB, specifically, we can use the redshift range of objects which are classified as galaxies within the target area. A further filter by V magnitude will also be applied.


In [ ]:
mask = gals_df['Redshift'].notna()
logger.debug(f'z range: {min(gals_df.loc[mask, 'Redshift'])} < z < {max(gals_df.loc[mask, 'Redshift'])}')
logger.debug(f'Invalid z entries: {len(gals_df.loc[gals_df['Redshift'].isna(), 'Redshift'])}')
logger.debug(f'RA range: {min(gals_df['ra']) * u.deg:.5f} < RA <  {max(gals_df['ra']) * u.deg:.5f}')
logger.debug(f'Dec range: {min(gals_df['dec']) * u.deg:.5f} < Dec <  {max(gals_df['dec']) * u.deg:.5f}')



In [6]:
from astroquery.simbad import Simbad

adql = """
SELECT DISTINCT filter
FROM basic AS b
JOIN flux AS f ON b.oid = f.oidref
WHERE b.ra  BETWEEN 194.76933 AND 195.28932
  AND b.dec BETWEEN  27.82581 AND  28.12152
  AND b.otype = 'G..'
  AND b.rvz_redshift BETWEEN 0.01241 AND 0.044067
ORDER BY filter
"""

tbl = Simbad.query_tap(adql)         # Astropy Table
df_filters = tbl.to_pandas()                 # -> pandas DataFrame
df_filters


,filter
0,B
1,g
2,G
3,H
4,i
5,I
6,J
7,K
8,r
9,R


In [7]:
from astroquery.simbad import Simbad

# adql = """
# SELECT
#   b.main_id,
#   b.ra, b.dec,
#   b.otype,
#   b.rvz_redshift AS z,
#   f."V" AS Vmag
# FROM basic AS b
# JOIN allfluxes AS f ON b.oid = f.oidref
# WHERE b.ra  BETWEEN 194.76933 AND 195.28932
#   AND b.dec BETWEEN  27.82581 AND  28.12152
#   AND b.otype = 'G..'                      -- galaxies & subtypes
#   AND b.rvz_redshift BETWEEN 0.01241 AND 0.044067
#   AND f."V" < 20
# ORDER BY Vmag ASC
# """

adql = """
SELECT
  b.main_id,
  b.ra, b.dec,
  b.otype,
  b.rvz_redshift AS z,

  fV.flux AS Vmag,

  fB.flux AS Bmag,
  (fB.flux - fV.flux) AS BV,

  fg.flux AS gmag,
  fr.flux AS rmag,
  fi.flux AS imag,
  fz.flux AS zmag,

  (fg.flux - fi.flux) AS gi,
  (fg.flux - fr.flux) AS gr

FROM basic AS b

-- define your "sample membership" magnitude cut using V
JOIN flux AS fV
  ON b.oid = fV.oidref AND fV.filter = 'V'

-- optional joins: only present if that filter exists for the object
LEFT JOIN flux AS fB
  ON b.oid = fB.oidref AND fB.filter = 'B'
LEFT JOIN flux AS fg
  ON b.oid = fg.oidref AND fg.filter = 'g'
LEFT JOIN flux AS fr
  ON b.oid = fr.oidref AND fr.filter = 'r'
LEFT JOIN flux AS fi
  ON b.oid = fi.oidref AND fi.filter = 'i'
LEFT JOIN flux AS fz
  ON b.oid = fz.oidref AND fz.filter = 'z'

WHERE b.ra  BETWEEN 194.76933 AND 195.28932
  AND b.dec BETWEEN  27.82581 AND  28.12152
  AND b.otype = 'G..'
  AND b.rvz_redshift BETWEEN 0.01241 AND 0.044067
  AND fV.flux < 20

ORDER BY Vmag ASC
"""


tbl = Simbad.query_tap(adql)         # Astropy Table
df = tbl.to_pandas()                 # -> pandas DataFrame
df


,main_id,ra,dec,otype,z,vmag,bmag,bv,gmag,rmag,imag,zmag,gi,gr
0,NGC 4889,195.033738,27.977025,EmG,0.021500,11.300000,13.00,1.700000,12.352800,11.5017,11.527000,10.834600,0.825800,0.851100
1,NGC 4874,194.898789,27.959248,LIN,0.023910,12.710000,13.70,0.990000,12.647200,11.7979,11.660000,11.153500,0.987200,0.849299
2,IC 4051,195.226929,28.007639,EmG,0.016620,13.500000,14.80,1.300000,13.830000,13.0450,12.617000,12.289000,1.213000,0.785000
3,NGC 4869,194.847328,27.911592,EmG,0.022879,13.520000,14.90,1.379999,14.429000,13.6240,13.195000,12.914000,1.234000,0.805000
4,NGC 4908,195.214762,28.042874,LIN,0.029160,13.580000,14.90,1.320000,14.449800,13.6108,13.173000,12.884000,1.276799,0.839000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
203,GMP 2707,195.097850,28.050510,G,0.023603,19.889999,20.60,0.710001,20.480000,NaN,NaN,NaN,NaN,NaN
204,SDSS J130005.76+280212.1,195.024015,28.036713,G,0.019400,19.910000,NaN,NaN,20.724001,20.1980,19.941999,20.280001,0.782001,0.526001
205,SDSS J130016.67+275638.6,195.069472,27.944075,LSB,0.017899,19.930000,20.58,0.650000,20.440001,21.2400,21.000000,20.900000,-0.559999,-0.799999
206,SDSS J125934.40+275942.8,194.893346,27.995247,G,0.017429,19.940001,NaN,NaN,22.070000,21.3400,21.100000,21.299999,0.969999,0.730000


In [10]:
# SIMBAD + NED with galaxy dimensions & V<20
from astroquery.simbad import Simbad
from astroquery.ipac.ned import Ned
import pandas as pd
import numpy as np

# --- 1) Pull the base sample from SIMBAD TAP (V-band via allfluxes, z via rvz_redshift)
adql = """
SELECT
  b.main_id,
  b.ra, b.dec,                  -- degrees (ICRS)
  b.otype,
  b.rvz_redshift AS z,
  fV.flux AS Vmag_simbad,               -- V magnitude

  fB.flux AS Bmag_simbad,
  (fB.flux - fV.flux) AS BV_simbad,

  fg.flux AS gmag_simbad,
  fr.flux AS rmag_simbad,
  fi.flux AS imag_simbad,
  fz.flux AS zmag_simbad,

  (fg.flux - fi.flux) AS gi_simbad,
  (fg.flux - fr.flux) AS gr_simbad

FROM basic AS b

-- define your "sample membership" magnitude cut using V
JOIN flux AS fV
  ON b.oid = fV.oidref AND fV.filter = 'V'

-- optional joins: only present if that filter exists for the object
LEFT JOIN flux AS fB
  ON b.oid = fB.oidref AND fB.filter = 'B'
LEFT JOIN flux AS fg
  ON b.oid = fg.oidref AND fg.filter = 'g'
LEFT JOIN flux AS fr
  ON b.oid = fr.oidref AND fr.filter = 'r'
LEFT JOIN flux AS fi
  ON b.oid = fi.oidref AND fi.filter = 'i'
LEFT JOIN flux AS fz
  ON b.oid = fz.oidref AND fz.filter = 'z'

WHERE b.ra  BETWEEN 194.76933 AND 195.28932
  AND b.dec BETWEEN  27.82581 AND  28.12152
  AND b.otype = 'G..'                      -- galaxies & subtypes
  AND b.rvz_redshift BETWEEN 0.01241 AND 0.044067
  AND fV.flux < 19
"""

base = Simbad.query_tap(adql)          # Astropy Table
df = base.to_pandas()

# --- 2) Batch-fetch galaxy dimensions from SIMBAD (major/minor/PA).
#       These arrive as arcminutes (at B=25 isophote), column names: galdim_majaxis, ...
s = Simbad()
s.add_votable_fields("dim")            # -> galdim_majaxis, galdim_minaxis, galdim_angle, ...
dims = s.query_objects(df["main_id"].tolist()).to_pandas()

# Keep only the columns we need and make sure they’re unique by main_id
keep = ["main_id", "Vmag_simbad", "Bmag_simbad", "BV_simbad", "gmag_simbad", "rmag_simbad", "imag_simbad", "zmag_simbad", "gr_simbad", "gi_simbad", "galdim_majaxis", "galdim_minaxis", "galdim_angle", "galdim_qual", "galdim_bibcode"]
dims = dims[[c for c in keep if c in dims.columns]].drop_duplicates(subset="main_id", keep="first")

# Merge onto the base sample
out = df.merge(dims, on="main_id", how="left")

# Convert SIMBAD major axis (arcmin) -> arcsec and tag the source
out["major_axis_arcsec"] = out.get("galdim_majaxis") * 60.0
out["dim_source"] = np.where(out["major_axis_arcsec"].notna(), "SIMBAD", None)

# --- 3) Fallback to NED for rows still missing a major axis
def ned_major_axis_arcsec(name: str):
    """Return a best-effort major-axis in arcsec from NED 'diameters' table (or np.nan)."""
    try:
        t = Ned.get_table(name, table='diameters')  # may raise or return empty
    except Exception:
        return np.nan, None, None  # value, pa_deg, refcode

    if t is None or len(t) == 0:
        return np.nan, None, None

    # Try to find a column with 'major' (prefer one that mentions arcsec)
    maj_cols = [c for c in t.colnames if 'major' in c.lower()]
    if not maj_cols:
        return np.nan, None, None
    # Prefer a column explicitly labeled in arcsec if present
    cmaj = next((c for c in maj_cols if 'arcsec' in c.lower() or '(")' in c.lower()), maj_cols[0])

    # NED often has multiple rows (different bands/methods). Take the largest numeric value.
    vals = pd.to_numeric(pd.Series(t[cmaj]), errors='coerce')
    if vals.notna().any():
        idx = int(vals.idxmax())
        val = float(vals.iloc[vals.argmax()])
        # Try to also grab PA and a reference code if present
        pa_col = next((c for c in t.colnames if 'pa' in c.lower() and 'deg' in c.lower()), None)
        ref_col = next((c for c in t.colnames if 'ref' in c.lower()), None)
        pa = float(pd.to_numeric(pd.Series(t[pa_col]), errors='coerce').iloc[idx]) if pa_col else None
        refcode = str(t[ref_col][idx]) if ref_col else None
        return val, pa, refcode

    return np.nan, None, None

mask = out["major_axis_arcsec"].isna()
if mask.any():
    fill_vals = []
    fill_pa = []
    fill_ref = []
    for name in out.loc[mask, "main_id"]:
        v, pa, refcode = ned_major_axis_arcsec(name)
        fill_vals.append(v)
        fill_pa.append(pa)
        fill_ref.append(refcode)

    # Assign back to the corresponding rows
    idx_mask = out.index[mask]  # index positions we attempted to fill
    
    # Make sure columns exist
    if "dim_ref" not in out.columns:
        out["dim_ref"] = pd.NA
    if "galdim_angle" not in out.columns:
        out["galdim_angle"] = np.nan
    
    # Build Series aligned to the masked index
    s_major = pd.Series(fill_vals, index=idx_mask, dtype="float64")
    s_pa    = pd.Series(fill_pa,   index=idx_mask, dtype="float64")
    s_ref   = pd.Series(fill_ref,  index=idx_mask, dtype="object")
    
    # Assign back exactly to those rows (lengths now match)
    out["major_axis_arcsec"] = out["major_axis_arcsec"].astype("float64")
    out.loc[idx_mask, "major_axis_arcsec"] = s_major.to_numpy(dtype="float64")
    # out.loc[idx_mask, "major_axis_arcsec"] = s_major.values
    out.loc[idx_mask, "galdim_angle"]      = s_pa.values
    out.loc[idx_mask, "dim_ref"]           = s_ref.values
    
    # Mark source where we actually obtained a value from NED
    out.loc[idx_mask, "dim_source"] = np.where(
        s_major.notna(),
        "NED",
        out.loc[idx_mask, "dim_source"]  # keep existing (e.g., None) if still NaN
    )

# Tidy: order columns
cols = ["main_id", "ra", "dec", "otype", "z", "vmag_simbad", "bmag_simbad", "bv_simbad", "gmag_simbad", "rmag_simbad", "imag_simbad", "zmag_simbad", "gr_simbad", "gi_simbad", 
        "major_axis_arcsec", "galdim_minaxis", "galdim_angle", "dim_source", "galdim_qual", "galdim_bibcode", "dim_ref"]
out = out[[c for c in cols if c in out.columns]]

out['MV_simbad'] = out['vmag_simbad'] - 35

# Show the DataFrame
out


,main_id,ra,dec,otype,z,vmag_simbad,bmag_simbad,bv_simbad,gmag_simbad,rmag_simbad,...,gr_simbad,gi_simbad,major_axis_arcsec,galdim_minaxis,galdim_angle,dim_source,galdim_qual,galdim_bibcode,dim_ref,MV_simbad
0,SDSS J125939.47+275116.5,194.914488,27.854598,G,0.031340,16.944000,17.816999,0.872999,16.851999,16.194000,...,0.657999,0.967999,1.300000,NaN,<NA>,NED,,,2007SDSS6.C...0000:,-18.056000
1,LEDA 126771,195.015438,27.964525,GiC,0.017780,17.260000,18.490000,1.230000,17.207001,16.400999,...,0.806002,1.181002,7.150000,NaN,<NA>,NED,,,2007SDSS6.C...0000:,-17.740000
2,SDSS J130033.33+275849.3,195.138894,27.980364,AG?,0.017200,17.930000,19.882999,1.952999,19.080999,18.424999,...,0.656000,1.035000,5.600000,NaN,<NA>,NED,,,2007SDSS6.C...0000:,-17.070000
3,2MASX J12592536+2756038,194.855579,27.934492,GiC,0.025310,17.020000,18.094000,1.073999,17.577999,16.871000,...,0.706999,1.028000,8.580000,0.143,90,SIMBAD,C,2006AJ....131.1163S,<NA>,-17.980000
4,GMP 2550,195.162417,28.009806,G,0.019787,18.450001,19.149000,0.698999,18.840000,NaN,...,NaN,NaN,5.640000,NaN,<NA>,NED,,,2007SDSS6.C...0000:,-16.549999
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
168,LEDA 1821892,195.143442,27.934682,G,0.029230,16.809999,18.511999,1.702000,17.386000,16.688999,...,0.697001,0.962000,12.599999,0.190,<NA>,SIMBAD,E,2003A&A...412...45P,<NA>,-18.190001
169,2MASX J13004540+2750076,195.189196,27.835456,GiC,0.029270,16.709999,18.048000,1.338001,17.603001,16.898001,...,0.705000,1.041000,10.020000,0.103,115,SIMBAD,C,2006AJ....131.1163S,<NA>,-18.290001
170,2MASX J12595013+2755292,194.958773,27.924850,GiC,0.032800,16.049999,17.563000,1.513000,16.698999,15.917000,...,0.782000,1.148999,12.000000,0.152,130,SIMBAD,C,2006AJ....131.1163S,<NA>,-18.950001
171,2MASX J13001027+2751502,195.042654,27.863970,GiC,0.033110,15.540000,17.103001,1.563001,16.517000,15.931000,...,0.586000,0.878000,14.580000,0.195,70,SIMBAD,C,2006AJ....131.1163S,<NA>,-19.459999


### Try to get homogeneous Vmag data from Hyperleda

This next cell was a test, but in the end the coverage was sparse for all but the bright galaxies in the Hyperleda DB... so, moving on...

In [11]:
out.columns

Index(['main_id', 'ra', 'dec', 'otype', 'z', 'vmag_simbad', 'bmag_simbad',
       'bv_simbad', 'gmag_simbad', 'rmag_simbad', 'imag_simbad', 'zmag_simbad',
       'gr_simbad', 'gi_simbad', 'major_axis_arcsec', 'galdim_minaxis',
       'galdim_angle', 'dim_source', 'galdim_qual', 'galdim_bibcode',
       'dim_ref', 'MV_simbad'],
      dtype='object')

In [ ]:
import numpy as np
import pandas as pd
from astroquery.xmatch import XMatch
import astropy.units as u

def xmatch_glade2_with_id(out, id_col="main_id", ra_col="ra", dec_col="dec", max_arcsec=2.0):
    """
    Crossmatch SIMBAD-selected sample (in out dataframe) to VizieR GLADE v2.3:
      VII/281/glade2

    Returns:
      xm (pandas.DataFrame): xmatch results (includes your input coords + GLADE cols)
    """
    cat1 = out[[id_col, ra_col, dec_col]].copy()
    cat1 = cat1.rename(columns={ra_col: "RA", dec_col: "DEC"})
    t1 = Table.from_pandas(cat1)

    xm = XMatch.query(
        cat1=t1,
        cat2="vizier:VII/281/glade2",
        max_distance=max_arcsec * u.arcsec,
        colRA1="RA",
        colDec1="DEC"
    ).to_pandas()

    xm = xm.sort_values("angDist").drop_duplicates(subset=id_col, keep="first")
    return xm

xm = xmatch_glade2(out, max_arcsec=2.0)
print("Matched rows:", len(xm))
print("GLADE2 columns:", xm.columns.tolist())


In [ ]:
xm

In [ ]:
print(out.columns)


In [12]:
out['show_label'] = out["main_id"].astype("string").str.contains(
    r"(?i)\b(NGC|IC)\b", na=False
)


/var/folders/_d/2vszj0nd6532d6knr4c9lnrw0000gn/T/ipykernel_85522/914833855.py:1: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  out['show_label'] = out["main_id"].astype("string").str.contains(


In [13]:
out.sort_values(by='MV_simbad', ascending=False)


,main_id,ra,dec,otype,z,vmag_simbad,bmag_simbad,bv_simbad,gmag_simbad,rmag_simbad,...,gi_simbad,major_axis_arcsec,galdim_minaxis,galdim_angle,dim_source,galdim_qual,galdim_bibcode,dim_ref,MV_simbad,show_label
23,SDSS J130037.31+275441.0,195.155462,27.911388,G,0.020300,18.969999,NaN,NaN,20.280001,19.632000,...,0.925001,NaN,NaN,<NA>,None,,,None,-16.030001,False
71,SDSS J130005.34+275628.8,195.022289,27.941354,G,0.023700,18.910000,19.995001,1.085001,19.759001,19.106001,...,0.932001,NaN,NaN,<NA>,None,,,None,-16.090000,False
15,SDSS J130027.87+275916.2,195.116153,27.987850,G,0.032000,18.900000,NaN,NaN,20.139999,19.566000,...,0.955000,NaN,NaN,<NA>,None,,,None,-16.100000,False
17,SDSS J130042.51+280325.4,195.177164,28.057068,AG?,0.019403,18.889999,19.900000,1.010000,19.962999,19.353001,...,0.962000,4.380000,NaN,<NA>,NED,,,2007SDSS6.C...0000:,-16.110001,False
107,GMP 3308,194.904625,27.972333,G,0.025400,18.841999,19.827000,0.985001,NaN,NaN,...,NaN,1.630000,NaN,<NA>,NED,,,2007SDSS6.C...0000:,-16.158001,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,NGC 4908,195.214762,28.042874,LIN,0.029160,13.580000,14.900000,1.320000,14.449800,13.610800,...,1.276799,57.341999,0.666983,57,SIMBAD,B,2023ApJS..269....3M,<NA>,-21.420000,True
109,NGC 4869,194.847328,27.911592,EmG,0.022879,13.520000,14.900000,1.379999,14.429000,13.624000,...,1.234000,49.683781,0.690853,47,SIMBAD,B,2023ApJS..269....3M,<NA>,-21.480000,True
114,IC 4051,195.226929,28.007639,EmG,0.016620,13.500000,14.800000,1.300000,13.830000,13.045000,...,1.213000,79.127998,0.915775,103,SIMBAD,B,2023ApJS..269....3M,<NA>,-21.500000,True
106,NGC 4874,194.898789,27.959248,LIN,0.023910,12.710000,13.700000,0.990000,12.647200,11.797900,...,0.987200,140.404205,2.260740,63,SIMBAD,B,2023ApJS..269....3M,<NA>,-22.290001,True


In [14]:
# collapse runs of whitespace to a single space + trim ends
out["main_id"] = (
    out["main_id"]
      .astype("string")                 # keeps NA values cleanly
      .str.replace(r"\s+", " ", regex=True)
      .str.strip()
)
out["main_id"] = (
    out["main_id"].astype("string")
      .str.replace("\u00A0", " ", regex=False)  # NBSP -> normal space
      .str.replace(r"\s+", " ", regex=True)
      .str.strip()
)
assert not out["main_id"].str.contains(r"\s{2,}", na=False).any()


In [15]:
# out_row = out.loc[out['main_id'] == 'NGC 4889']
out_row = out.iloc[23]
out_row

main_id              SDSS J130037.31+275441.0
ra                                 195.155462
dec                                 27.911388
otype                                       G
z                                      0.0203
vmag_simbad                         18.969999
bmag_simbad                               NaN
bv_simbad                                 NaN
gmag_simbad                         20.280001
rmag_simbad                            19.632
imag_simbad                            19.355
zmag_simbad                         19.389999
gr_simbad                            0.648001
gi_simbad                            0.925001
major_axis_arcsec                         NaN
galdim_minaxis                            NaN
galdim_angle                             <NA>
dim_source                               None
galdim_qual                                  
galdim_bibcode                               
dim_ref                                  None
MV_simbad                         

In [16]:
out = out.rename(columns={
    "main_id": "name",
    "vmag_simbad": "Vmag_simbad",
    "bmag_simbad": "Bmag_simbad",
    "bv_simbad": "BV_simbad",
}, errors="raise")
out

,name,ra,dec,otype,z,Vmag_simbad,Bmag_simbad,BV_simbad,gmag_simbad,rmag_simbad,...,gi_simbad,major_axis_arcsec,galdim_minaxis,galdim_angle,dim_source,galdim_qual,galdim_bibcode,dim_ref,MV_simbad,show_label
0,SDSS J125939.47+275116.5,194.914488,27.854598,G,0.031340,16.944000,17.816999,0.872999,16.851999,16.194000,...,0.967999,1.300000,NaN,<NA>,NED,,,2007SDSS6.C...0000:,-18.056000,False
1,LEDA 126771,195.015438,27.964525,GiC,0.017780,17.260000,18.490000,1.230000,17.207001,16.400999,...,1.181002,7.150000,NaN,<NA>,NED,,,2007SDSS6.C...0000:,-17.740000,False
2,SDSS J130033.33+275849.3,195.138894,27.980364,AG?,0.017200,17.930000,19.882999,1.952999,19.080999,18.424999,...,1.035000,5.600000,NaN,<NA>,NED,,,2007SDSS6.C...0000:,-17.070000,False
3,2MASX J12592536+2756038,194.855579,27.934492,GiC,0.025310,17.020000,18.094000,1.073999,17.577999,16.871000,...,1.028000,8.580000,0.143,90,SIMBAD,C,2006AJ....131.1163S,<NA>,-17.980000,False
4,GMP 2550,195.162417,28.009806,G,0.019787,18.450001,19.149000,0.698999,18.840000,NaN,...,NaN,5.640000,NaN,<NA>,NED,,,2007SDSS6.C...0000:,-16.549999,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
168,LEDA 1821892,195.143442,27.934682,G,0.029230,16.809999,18.511999,1.702000,17.386000,16.688999,...,0.962000,12.599999,0.190,<NA>,SIMBAD,E,2003A&A...412...45P,<NA>,-18.190001,False
169,2MASX J13004540+2750076,195.189196,27.835456,GiC,0.029270,16.709999,18.048000,1.338001,17.603001,16.898001,...,1.041000,10.020000,0.103,115,SIMBAD,C,2006AJ....131.1163S,<NA>,-18.290001,False
170,2MASX J12595013+2755292,194.958773,27.924850,GiC,0.032800,16.049999,17.563000,1.513000,16.698999,15.917000,...,1.148999,12.000000,0.152,130,SIMBAD,C,2006AJ....131.1163S,<NA>,-18.950001,False
171,2MASX J13001027+2751502,195.042654,27.863970,GiC,0.033110,15.540000,17.103001,1.563001,16.517000,15.931000,...,0.878000,14.580000,0.195,70,SIMBAD,C,2006AJ....131.1163S,<NA>,-19.459999,False


In [17]:
out['Re_arcsec'] = out['major_axis_arcsec'] / 8
out

,name,ra,dec,otype,z,Vmag_simbad,Bmag_simbad,BV_simbad,gmag_simbad,rmag_simbad,...,major_axis_arcsec,galdim_minaxis,galdim_angle,dim_source,galdim_qual,galdim_bibcode,dim_ref,MV_simbad,show_label,Re_arcsec
0,SDSS J125939.47+275116.5,194.914488,27.854598,G,0.031340,16.944000,17.816999,0.872999,16.851999,16.194000,...,1.300000,NaN,<NA>,NED,,,2007SDSS6.C...0000:,-18.056000,False,0.16250
1,LEDA 126771,195.015438,27.964525,GiC,0.017780,17.260000,18.490000,1.230000,17.207001,16.400999,...,7.150000,NaN,<NA>,NED,,,2007SDSS6.C...0000:,-17.740000,False,0.89375
2,SDSS J130033.33+275849.3,195.138894,27.980364,AG?,0.017200,17.930000,19.882999,1.952999,19.080999,18.424999,...,5.600000,NaN,<NA>,NED,,,2007SDSS6.C...0000:,-17.070000,False,0.70000
3,2MASX J12592536+2756038,194.855579,27.934492,GiC,0.025310,17.020000,18.094000,1.073999,17.577999,16.871000,...,8.580000,0.143,90,SIMBAD,C,2006AJ....131.1163S,<NA>,-17.980000,False,1.07250
4,GMP 2550,195.162417,28.009806,G,0.019787,18.450001,19.149000,0.698999,18.840000,NaN,...,5.640000,NaN,<NA>,NED,,,2007SDSS6.C...0000:,-16.549999,False,0.70500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
168,LEDA 1821892,195.143442,27.934682,G,0.029230,16.809999,18.511999,1.702000,17.386000,16.688999,...,12.599999,0.190,<NA>,SIMBAD,E,2003A&A...412...45P,<NA>,-18.190001,False,1.57500
169,2MASX J13004540+2750076,195.189196,27.835456,GiC,0.029270,16.709999,18.048000,1.338001,17.603001,16.898001,...,10.020000,0.103,115,SIMBAD,C,2006AJ....131.1163S,<NA>,-18.290001,False,1.25250
170,2MASX J12595013+2755292,194.958773,27.924850,GiC,0.032800,16.049999,17.563000,1.513000,16.698999,15.917000,...,12.000000,0.152,130,SIMBAD,C,2006AJ....131.1163S,<NA>,-18.950001,False,1.50000
171,2MASX J13001027+2751502,195.042654,27.863970,GiC,0.033110,15.540000,17.103001,1.563001,16.517000,15.931000,...,14.580000,0.195,70,SIMBAD,C,2006AJ....131.1163S,<NA>,-19.459999,False,1.82250


In [18]:
# manual row for cluster, for overall centering and plotting
manual = {
    "name": "Coma cluster",
    "ra": 195.01707133,               # center RA adjusted from BCG NGC 4889
    "dec": 27.977025,                 # center Dec too
    "otype": "CLUSTER",
    "z": 0.0215,                      # again, a representative z from NGC 4889
    # "Vmag": pd.NA,
    # "major_axis_arcsec": pd.NA,
    "dim_source": "MANUAL",
    "show_label": False
}

extra = pd.DataFrame([manual]).reindex(columns=out.columns, fill_value=pd.NA)
out = pd.concat([out, extra], ignore_index=True)


/var/folders/_d/2vszj0nd6532d6knr4c9lnrw0000gn/T/ipykernel_85522/2394890328.py:15: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  out = pd.concat([out, extra], ignore_index=True)


In [19]:
gals_df = out.copy()

In [20]:

# per-type counts
otype_counts = (
    gals_df["otype"].astype("string").str.strip()
      .value_counts(dropna=False)
      .rename_axis("otype")
      .reset_index(name="count")
)
otype_counts.head(10)


,otype,count
0,GiC,63
1,G,52
2,AG?,25
3,EmG,18
4,LIN,5
5,AGN,5
6,GiG,3
7,LSB,2
8,CLUSTER,1


In [21]:
# minimal, practical cheat-sheet (not exhaustive)
otype_map = {
    # single galaxies
    "G": "Galaxy",
    "G?": "Galaxy candidate",
    "LSB": "Low-surface-brightness galaxy",
    "BCD": "Blue compact dwarf galaxy",
    "dG": "Dwarf galaxy",
    "IG": "Interacting galaxy",
    "EmG": "Emission-line galaxy",
    # activity/AGN subclasses
    "SyG": "Seyfert galaxy",
    "Sy1": "Seyfert 1",
    "Sy2": "Seyfert 2",
    "LIN": "LINER galaxy",
    "AGN": "Active galactic nucleus (galaxy)",
    "BLLac": "BL Lac object",
    "QSO": "Quasar",
    # membership in larger structures
    "GiP": "Galaxy in a pair",
    "GiG": "Galaxy in a group",
    "GiC": "Galaxy in a cluster",
    "BiC": "Brightest cluster galaxy (BCG)",
    # systems of galaxies
    "PaG": "Pair of galaxies",
    "GrG": "Group of galaxies",
    "CGG": "Compact group of galaxies",
    "ClG": "Cluster of galaxies",
    "SCG": "Supercluster of galaxies",
    # broad morphology (when present in otype)
    "E": "Elliptical galaxy",
    "S0": "Lenticular galaxy",
    "S": "Spiral galaxy",
    "SA": "Unbarred spiral",
    "SB": "Barred spiral",
    "SAB": "Weakly barred spiral",
    "Im": "Irregular galaxy",
    # alternates you might also see
    "GPair": "Galaxy pair",
    "GTrpl": "Galaxy triple",
    "GGroup": "Galaxy group",
}

lookup = (pd.DataFrame.from_dict(otype_map, orient="index", columns=["meaning"])
          .reset_index().rename(columns={"index": "otype"}))

otype_counts = otype_counts.merge(lookup, on="otype", how="left")
otype_counts.head(20)


,otype,count,meaning
0,GiC,63,Galaxy in a cluster
1,G,52,Galaxy
2,AG?,25,NaN
3,EmG,18,Emission-line galaxy
4,LIN,5,LINER galaxy
5,AGN,5,Active galactic nucleus (galaxy)
6,GiG,3,Galaxy in a group
7,LSB,2,Low-surface-brightness galaxy
8,CLUSTER,1,NaN


In [22]:
family_bins = {
    "Active (AGN/Seyfert/QSO)": {"AGN","SyG","Sy1","Sy2","LIN","BLLac","QSO"},
    "Systems (pairs/groups/clusters)": {"PaG","GrG","CGG","ClG","SCG","GPair","GTrpl","GGroup"},
    "Members of systems": {"GiP","GiG","GiC","BiC"},
    "LSB/BCD/Dwarf": {"LSB","BCD","dG"},
    "Regular galaxies": {"G","E","S0","S","SA","SB","SAB","Im","IG","EmG"},
}
rev = {code: fam for fam, codes in family_bins.items() for code in codes}
gals_df["otype_family"] = gals_df["otype"].map(rev).fillna("Other")

family_counts = (gals_df["otype_family"]
                 .value_counts()
                 .rename_axis("otype_family")
                 .reset_index(name="count"))
family_counts


,otype_family,count
0,Regular galaxies,70
1,Members of systems,66
2,Other,26
3,Active (AGN/Seyfert/QSO),10
4,LSB/BCD/Dwarf,2


In [23]:
from astroquery.simbad import Simbad
s = Simbad(); 
s.add_votable_fields("morphtype")

morph = s.query_objects(gals_df["name"].tolist()).to_pandas()[["main_id","morph_type"]] 
morph = morph.rename(columns={"main_id": "name"})
morph
gals_df = gals_df.merge(morph, on="name", how="left")


In [24]:
# per-type counts
morphtype_counts = (
    gals_df["morph_type"].astype("string").str.strip()
      .value_counts(dropna=False)
      .rename_axis("morph_type")
      .reset_index(name="count")
)
morphtype_counts


,morph_type,count
0,<NA>,68
1,,28
2,dE,11
3,SB0,11
4,S0,5
5,Sa,5
6,dE...,4
7,S...,4
8,SBa,4
9,E,4


In [25]:
is_E   = (gals_df["otype"] == "E")
is_BCG = (gals_df["otype"] == "BiC")  # SIMBAD code for brightest cluster galaxy
has_cD = gals_df["morph_type"].astype("string").str.contains(r"\bcD\b", na=False)


In [26]:
is_E = ~(gals_df["otype"].astype("string").str.contains(r"CLUSTER", na=False)) 
is_E = ~(gals_df["morph_type"].astype("string").str.contains(r"S", na=False)) & is_E
# is_E = (gals_df["morph_type"].astype("string")=="I") | is_E



In [27]:
is_E = (gals_df["morph_type"]!="I") & is_E


In [28]:
gals_df[is_E]

,name,ra,dec,otype,z,Vmag_simbad,Bmag_simbad,BV_simbad,gmag_simbad,rmag_simbad,...,galdim_angle,dim_source,galdim_qual,galdim_bibcode,dim_ref,MV_simbad,show_label,Re_arcsec,otype_family,morph_type
0,SDSS J125939.47+275116.5,194.914488,27.854598,G,0.03134,16.944000,17.816999,0.872999,16.851999,16.194000,...,<NA>,NED,,,2007SDSS6.C...0000:,-18.056000,False,0.162500,Regular galaxies,
1,LEDA 126771,195.015438,27.964525,GiC,0.01778,17.260000,18.490000,1.230000,17.207001,16.400999,...,<NA>,NED,,,2007SDSS6.C...0000:,-17.740000,False,0.893750,Members of systems,NaN
2,SDSS J130033.33+275849.3,195.138894,27.980364,AG?,0.01720,17.930000,19.882999,1.952999,19.080999,18.424999,...,<NA>,NED,,,2007SDSS6.C...0000:,-17.070000,False,0.700000,Other,
5,SDSS J125942.36+280158.5,194.926540,28.032920,G,0.02630,18.760000,NaN,NaN,NaN,NaN,...,<NA>,NED,,,2007SDSS6.C...0000:,-16.240000,False,0.876250,Regular galaxies,
7,SDSS J125926.45+275124.7,194.860237,27.856882,AG?,0.01660,17.719999,19.120001,1.400002,18.222000,17.548000,...,<NA>,NED,,,2007SDSS6.C...0000:,-17.280001,False,0.712500,Other,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
164,LEDA 126781,194.945554,27.991910,G,0.02806,17.430000,18.620001,1.190001,18.250999,17.563000,...,<NA>,NED,,,2007SDSS6.C...0000:,-17.570000,False,0.913750,Regular galaxies,NaN
165,LEDA 44809,195.178493,27.963119,EmG,0.02809,14.960000,16.013000,1.053000,15.470000,14.708000,...,93,SIMBAD,B,2023ApJS..269....3M,<NA>,-20.040001,False,4.806052,Regular galaxies,NaN
167,SDSS J130041.19+280242.4,195.171675,28.045127,G,0.02877,17.150000,18.487000,1.337000,18.054001,17.327999,...,<NA>,NED,,,2007SDSS6.C...0000:,-17.850000,False,0.883750,Regular galaxies,E3
168,LEDA 1821892,195.143442,27.934682,G,0.02923,16.809999,18.511999,1.702000,17.386000,16.688999,...,<NA>,SIMBAD,E,2003A&A...412...45P,<NA>,-18.190001,False,1.575000,Regular galaxies,dG


In [29]:
len(gals_df[is_E==True])


132

In [30]:
len(gals_df[is_BCG==True])


0

In [31]:
len(gals_df[has_cD==True])


0

In [32]:
from astropy.cosmology import Planck15 as cosmo
import numpy as np

# distance modulus & angular diameter distance
z = gals_df["z"].astype(float)
DM = cosmo.distmod(z).value                 # mag
DM = 35.0  # assumed value 
DA = cosmo.angular_diameter_distance(z).to("kpc").value  # kpc

# absolute M_V (ignore extinction unless you have it)
# gals_df["M_V"] = gals_df["Vmag"].astype(float) - DM

# crude physical size from your major-axis measurement if present (arcsec → kpc)
maj_arcsec = pd.to_numeric(gals_df.get("major_axis_arcsec"), errors="coerce")
gals_df["major_kpc"] = (maj_arcsec/206265.0) * (DA*1e3)  # 1 rad = 206265"

is_luminous = gals_df["MV_simbad"] <= -21.5
is_huge     = gals_df["major_kpc"] >= 30                  # very extended on the sky


In [33]:
gals_df[is_luminous]

,name,ra,dec,otype,z,Vmag_simbad,Bmag_simbad,BV_simbad,gmag_simbad,rmag_simbad,...,dim_source,galdim_qual,galdim_bibcode,dim_ref,MV_simbad,show_label,Re_arcsec,otype_family,morph_type,major_kpc
96,NGC 4889,195.033738,27.977025,EmG,0.02150,11.30,13.0,1.70,12.3528,11.5017,...,SIMBAD,B,2023ApJS..269....3M,<NA>,-23.700001,True,21.925575,Regular galaxies,NaN,78815.629278
106,NGC 4874,194.898789,27.959248,LIN,0.02391,12.71,13.7,0.99,12.6472,11.7979,...,SIMBAD,B,2023ApJS..269....3M,<NA>,-22.290001,True,17.550526,Active (AGN/Seyfert/QSO),NaN,69955.709614
114,IC 4051,195.226929,28.007639,EmG,0.01662,13.50,14.8,1.30,13.8300,13.0450,...,SIMBAD,B,2023ApJS..269....3M,<NA>,-21.500000,True,9.891000,Regular galaxies,E3,27648.455901


In [34]:
gals_df[is_huge]

,name,ra,dec,otype,z,Vmag_simbad,Bmag_simbad,BV_simbad,gmag_simbad,rmag_simbad,...,dim_source,galdim_qual,galdim_bibcode,dim_ref,MV_simbad,show_label,Re_arcsec,otype_family,morph_type,major_kpc
0,SDSS J125939.47+275116.5,194.914488,27.854598,G,0.031340,16.944000,17.816999,0.872999,16.851999,16.194000,...,NED,,,2007SDSS6.C...0000:,-18.056000,False,0.16250,Regular galaxies,,841.404907
1,LEDA 126771,195.015438,27.964525,GiC,0.017780,17.260000,18.490000,1.230000,17.207001,16.400999,...,NED,,,2007SDSS6.C...0000:,-17.740000,False,0.89375,Members of systems,NaN,2668.911659
2,SDSS J130033.33+275849.3,195.138894,27.980364,AG?,0.017200,17.930000,19.882999,1.952999,19.080999,18.424999,...,NED,,,2007SDSS6.C...0000:,-17.070000,False,0.70000,Other,,2023.575699
3,2MASX J12592536+2756038,194.855579,27.934492,GiC,0.025310,17.020000,18.094000,1.073999,17.577999,16.871000,...,SIMBAD,C,2006AJ....131.1163S,<NA>,-17.980000,False,1.07250,Members of systems,E/S0,4517.585881
4,GMP 2550,195.162417,28.009806,G,0.019787,18.450001,19.149000,0.698999,18.840000,NaN,...,NED,,,2007SDSS6.C...0000:,-16.549999,False,0.70500,Regular galaxies,Sa,2337.200564
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
168,LEDA 1821892,195.143442,27.934682,G,0.029230,16.809999,18.511999,1.702000,17.386000,16.688999,...,SIMBAD,E,2003A&A...412...45P,<NA>,-18.190001,False,1.57500,Regular galaxies,dG,7625.491315
169,2MASX J13004540+2750076,195.189196,27.835456,GiC,0.029270,16.709999,18.048000,1.338001,17.603001,16.898001,...,SIMBAD,C,2006AJ....131.1163S,<NA>,-18.290001,False,1.25250,Members of systems,I,6072.086296
170,2MASX J12595013+2755292,194.958773,27.924850,GiC,0.032800,16.049999,17.563000,1.513000,16.698999,15.917000,...,SIMBAD,C,2006AJ....131.1163S,<NA>,-18.950001,False,1.50000,Members of systems,S0...,8114.346441
171,2MASX J13001027+2751502,195.042654,27.863970,GiC,0.033110,15.540000,17.103001,1.563001,16.517000,15.931000,...,SIMBAD,C,2006AJ....131.1163S,<NA>,-19.459999,False,1.82250,Members of systems,SB0,9948.394156


In [35]:
gals_df["is_large_E"] = (is_E & (is_luminous | is_huge)) | is_BCG | has_cD


In [59]:
import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord, match_coordinates_sky
import astropy.units as u
from astroquery.sdss import SDSS

def attach_sdss_re(
    gals_df, ra_col="ra", dec_col="dec",
    band="r", radius="3 arcsec", dr=17,
    add_phot=True, store_modelmag=True,
    use_extcorr=True,
    add_vmag=True
):
    df = gals_df.copy()
    coords = SkyCoord(df[ra_col].values, df[dec_col].values, unit="deg", frame="icrs")

    # --- SDSS CrossID fields ---
    fields = [
        f"deVRad_{band}", f"deVAB_{band}",
        f"expRad_{band}", f"expAB_{band}",
        f"fracDeV_{band}",
        "type", "mode", "objid", "ra", "dec",
    ]

    if add_phot:
        # cModel (recommended for "total-ish" galaxy mags)
        fields += [
            "cModelMag_g", "cModelMag_r", "cModelMag_i",
            "cModelMagErr_g", "cModelMagErr_r", "cModelMagErr_i",
            "extinction_g", "extinction_r", "extinction_i",
        ]
        if store_modelmag:
            # modelMag (often used for colors; you can decide later)
            fields += [
                "modelMag_g", "modelMag_r", "modelMag_i",
                "modelMagErr_g", "modelMagErr_r", "modelMagErr_i",
            ]

    xid = SDSS.query_crossid(coords, radius=radius, photoobj_fields=fields, data_release=dr)
    if xid is None or len(xid) == 0:
        # sizes
        for c in ["Re_r_arcsec", "Re_r_arcsec_circ", "sdss_objid"]:
            df[c] = pd.NA
        # phot
        if add_phot:
            for c in [
                "g_cmodel_sdss", "r_cmodel_sdss", "i_cmodel_sdss",
                "gerr_cmodel_sdss", "rerr_cmodel_sdss", "ierr_cmodel_sdss",
                "Ag_sdss", "Ar_sdss", "Ai_sdss",
                "g0_cmodel_sdss", "r0_cmodel_sdss", "i0_cmodel_sdss",
                "gr0_sdss", "gi0_sdss",
                "sdss_phot_ok",
            ]:
                df[c] = pd.NA
            if store_modelmag:
                for c in [
                    "g_model_sdss", "r_model_sdss", "i_model_sdss",
                    "gerr_model_sdss", "rerr_model_sdss", "ierr_model_sdss",
                    "g0_model_sdss", "r0_model_sdss", "i0_model_sdss",
                ]:
                    df[c] = pd.NA
        if add_vmag:
            for c in ["Vmag_sdss", "Vmag_sdss_err", "V_source"]:
                df[c] = pd.NA
        return df

    t = xid.to_pandas()

    # Prefer primary galaxy detections if columns exist
    if "mode" in t.columns:
        t = t[t["mode"] == 1]
    if "type" in t.columns:
        t = t[t["type"] == 3]  # 3=GALAXY

    # Match returned SDSS rows back to input rows by nearest on-sky
    sdss_sc = SkyCoord(t["ra"].values, t["dec"].values, unit="deg")
    idx, sep, _ = match_coordinates_sky(coords, sdss_sc)

    max_sep = u.Quantity(radius)
    good = sep <= max_sep

    sel = t.iloc[idx].reset_index(drop=True)
    sel.loc[~good, :] = np.nan  # blank out non-matches

    # ---- Sizes (your existing logic) ----
    frac = sel.get(f"fracDeV_{band}")
    Re_deV = sel.get(f"deVRad_{band}")
    Re_exp = sel.get(f"expRad_{band}")

    use_deV = frac.notna() & (frac >= 0.5)
    Re = np.where(use_deV, Re_deV, Re_exp)

    ba = np.where(use_deV, sel.get(f"deVAB_{band}"), sel.get(f"expAB_{band}"))
    Re_circ = Re * np.sqrt(ba)

    df["sdss_objid"]       = sel.get("objid").values
    df["Re_r_arcsec"]      = pd.to_numeric(Re, errors="coerce")
    df["Re_r_arcsec_circ"] = pd.to_numeric(Re_circ, errors="coerce")

    # ---- Photometry storage + colors ----
    if add_phot:
        # Raw cModel mags + errors
        df["g_cmodel_sdss"] = pd.to_numeric(sel.get("cModelMag_g"), errors="coerce")
        df["r_cmodel_sdss"] = pd.to_numeric(sel.get("cModelMag_r"), errors="coerce")
        df["i_cmodel_sdss"] = pd.to_numeric(sel.get("cModelMag_i"), errors="coerce")

        df["gerr_cmodel_sdss"] = pd.to_numeric(sel.get("cModelMagErr_g"), errors="coerce")
        df["rerr_cmodel_sdss"] = pd.to_numeric(sel.get("cModelMagErr_r"), errors="coerce")
        df["ierr_cmodel_sdss"] = pd.to_numeric(sel.get("cModelMagErr_i"), errors="coerce")

        # Extinctions
        df["Ag_sdss"] = pd.to_numeric(sel.get("extinction_g"), errors="coerce")
        df["Ar_sdss"] = pd.to_numeric(sel.get("extinction_r"), errors="coerce")
        df["Ai_sdss"] = pd.to_numeric(sel.get("extinction_i"), errors="coerce")

        if use_extcorr:
            df["g0_cmodel_sdss"] = df["g_cmodel_sdss"] - df["Ag_sdss"]
            df["r0_cmodel_sdss"] = df["r_cmodel_sdss"] - df["Ar_sdss"]
            df["i0_cmodel_sdss"] = df["i_cmodel_sdss"] - df["Ai_sdss"]
        else:
            df["g0_cmodel_sdss"] = df["g_cmodel_sdss"]
            df["r0_cmodel_sdss"] = df["r_cmodel_sdss"]
            df["i0_cmodel_sdss"] = df["i_cmodel_sdss"]

        # Bundle-present flag (all three bands exist)
        df["sdss_phot_ok"] = (
            df["g0_cmodel_sdss"].notna() &
            df["r0_cmodel_sdss"].notna() &
            df["i0_cmodel_sdss"].notna()
        )

        # Colors (only meaningful when bundle exists)
        df["gr0_sdss"] = df["g0_cmodel_sdss"] - df["r0_cmodel_sdss"]
        df["gi0_sdss"] = df["g0_cmodel_sdss"] - df["i0_cmodel_sdss"]
        df.loc[~df["sdss_phot_ok"], ["gr0_sdss", "gi0_sdss"]] = np.nan

        if store_modelmag:
            df["g_model_sdss"] = pd.to_numeric(sel.get("modelMag_g"), errors="coerce")
            df["r_model_sdss"] = pd.to_numeric(sel.get("modelMag_r"), errors="coerce")
            df["i_model_sdss"] = pd.to_numeric(sel.get("modelMag_i"), errors="coerce")

            df["gerr_model_sdss"] = pd.to_numeric(sel.get("modelMagErr_g"), errors="coerce")
            df["rerr_model_sdss"] = pd.to_numeric(sel.get("modelMagErr_r"), errors="coerce")
            df["ierr_model_sdss"] = pd.to_numeric(sel.get("modelMagErr_i"), errors="coerce")

            if use_extcorr:
                df["g0_model_sdss"] = df["g_model_sdss"] - df["Ag_sdss"]
                df["r0_model_sdss"] = df["r_model_sdss"] - df["Ar_sdss"]
                df["i0_model_sdss"] = df["i_model_sdss"] - df["Ai_sdss"]
            else:
                df["g0_model_sdss"] = df["g_model_sdss"]
                df["r0_model_sdss"] = df["r_model_sdss"]
                df["i0_model_sdss"] = df["i_model_sdss"]

    # ---- Vmag from SDSS g,r (optional) ----
    if add_vmag:
        # Use extinction-corrected cModel mags (consistent with your sdss_phot_ok)
        g = df["g0_cmodel_sdss"] if add_phot else pd.to_numeric(sel.get("cModelMag_g"), errors="coerce")
        r = df["r0_cmodel_sdss"] if add_phot else pd.to_numeric(sel.get("cModelMag_r"), errors="coerce")

        gerr = df["gerr_cmodel_sdss"] if add_phot else pd.to_numeric(sel.get("cModelMagErr_g"), errors="coerce")
        rerr = df["rerr_cmodel_sdss"] if add_phot else pd.to_numeric(sel.get("cModelMagErr_r"), errors="coerce")

        # V = 0.41 g + 0.59 r - 0.01
        df["Vmag_sdss"] = 0.41 * g + 0.59 * r - 0.01
        df["Vmag_sdss_err"] = np.sqrt((0.41 * gerr) ** 2 + (0.59 * rerr) ** 2)
        df.loc[df["Vmag_sdss"].isna(), "Vmag_sdss_err"] = np.nan

        df["V_source"] = np.where(df["Vmag_sdss"].notna(), "SDSS(cModelMag g,r)", pd.NA)

    return df



In [60]:
gals_df = attach_sdss_re(gals_df, ra_col="ra", dec_col="dec", band="r", radius="2 arcsec", dr=17, add_vmag=True)


In [61]:
gals_df.columns

Index(['name', 'ra', 'dec', 'otype', 'z', 'Vmag_simbad', 'Bmag_simbad',
       'BV_simbad', 'gmag_simbad', 'rmag_simbad', 'imag_simbad', 'zmag_simbad',
       'gr_simbad', 'gi_simbad', 'major_axis_arcsec', 'galdim_minaxis',
       'galdim_angle', 'dim_source', 'galdim_qual', 'galdim_bibcode',
       'dim_ref', 'MV_simbad', 'show_label', 'Re_arcsec', 'otype_family',
       'morph_type', 'major_kpc', 'is_large_E', 'sdss_objid', 'Re_r_arcsec',
       'Re_r_arcsec_circ', 'Vmag_sdss', 'Vmag_sdss_err', 'V_source', 'mV_app',
       'mV_src', 'MV_abs', 'Re_GCS_scale', 'Re_GCS_arcsec', 'n_gcs_prior',
       'n_gcs_red_prior', 'n_gcs_blue_prior', 'g_cmodel_sdss', 'r_cmodel_sdss',
       'i_cmodel_sdss', 'gerr_cmodel_sdss', 'rerr_cmodel_sdss',
       'ierr_cmodel_sdss', 'Ag_sdss', 'Ar_sdss', 'Ai_sdss', 'g0_cmodel_sdss',
       'r0_cmodel_sdss', 'i0_cmodel_sdss', 'sdss_phot_ok', 'gr0_sdss',
       'gi0_sdss', 'g_model_sdss', 'r_model_sdss', 'i_model_sdss',
       'gerr_model_sdss', 'rerr_model_sds

In [62]:
gals_df['Vmag_sdss'].describe()

count    162.000000
mean      16.770820
std        1.893492
min       11.899091
25%       15.413348
50%       16.897377
75%       17.994782
max       24.891213
Name: Vmag_sdss, dtype: float64

In [63]:
import numpy as np
import pandas as pd

def finalize_photometry(
    df,
    *,
    # V selection priority (first non-null wins)
    v_candidates=("Vmag_sdss", "Vmag_leda", "Vmag_simbad_raw", "Vmag_simbad"),
    v_labels=("SDSS", "LEDA", "SIMBAD", "SIMBAD"),
    # Use extinction-corrected SDSS colors if present
    sdss_g0="g0_cmodel_sdss",
    sdss_r0="r0_cmodel_sdss",
    sdss_i0="i0_cmodel_sdss",
    # Optional: also keep a “strict” ok flag based on bundle
    require_sdss_bundle=True,
    # If you already computed MV_abs elsewhere, leave it alone
    overwrite_existing=False,
):
    """
    Create canonical, analysis-safe photometry columns.

    Outputs (canonical):
      - mV_app, mV_src
      - gi_primary, gr_primary, color_src
      - sdss_phot_ok (or recomputed if missing)

    Also outputs helpful debug columns:
      - mV_app_col (which source column actually populated mV_app)
    """

    out = df.copy()

    # --- Ensure SIMBAD raw names are consistent (no-op if already done) ---
    # If both exist, prefer *_raw versions without overwriting.
    for base in ["Vmag_simbad", "Bmag_simbad", "BV_simbad", "gmag_simbad", "rmag_simbad", "imag_simbad", "zmag_simbad",
                 "gr_simbad", "gi_simbad", "MV_simbad"]:
        raw = f"{base}_raw"
        if raw not in out.columns and base in out.columns:
            out.rename(columns={base: raw}, inplace=True)

    # --- Canonical apparent V (mV_app) ---
    if overwrite_existing or ("mV_app" not in out.columns):
        out["mV_app"] = np.nan
    if overwrite_existing or ("mV_src" not in out.columns):
        out["mV_src"] = pd.NA
    if overwrite_existing or ("mV_app_col" not in out.columns):
        out["mV_app_col"] = pd.NA

    # Only fill where missing unless overwrite_existing=True
    need = out["mV_app"].isna() if not overwrite_existing else pd.Series(True, index=out.index)

    for col, lab in zip(v_candidates, v_labels):
        if col not in out.columns:
            continue
        mask = need & out[col].notna()
        if mask.any():
            out.loc[mask, "mV_app"] = pd.to_numeric(out.loc[mask, col], errors="coerce")
            out.loc[mask, "mV_src"] = lab
            out.loc[mask, "mV_app_col"] = col
            need = out["mV_app"].isna() if not overwrite_existing else pd.Series(False, index=out.index)

    # --- SDSS photometry bundle ok flag ---
    if "sdss_phot_ok" not in out.columns or overwrite_existing:
        out["sdss_phot_ok"] = (
            out.get(sdss_g0, pd.Series(np.nan, index=out.index)).notna() &
            out.get(sdss_r0, pd.Series(np.nan, index=out.index)).notna() &
            out.get(sdss_i0, pd.Series(np.nan, index=out.index)).notna()
        )

    # --- Canonical colors (SDSS only, no mixing) ---
    if overwrite_existing or ("gi_primary" not in out.columns):
        out["gi_primary"] = np.nan
    if overwrite_existing or ("gr_primary" not in out.columns):
        out["gr_primary"] = np.nan
    if overwrite_existing or ("color_src" not in out.columns):
        out["color_src"] = pd.NA

    # We only populate colors when SDSS bundle exists (unless you decide later to add BV fallback)
    ok = out["sdss_phot_ok"] if require_sdss_bundle else (
        out.get(sdss_g0, pd.Series(np.nan, index=out.index)).notna() &
        out.get(sdss_r0, pd.Series(np.nan, index=out.index)).notna()
    )

    g0 = pd.to_numeric(out.get(sdss_g0, np.nan), errors="coerce")
    r0 = pd.to_numeric(out.get(sdss_r0, np.nan), errors="coerce")
    i0 = pd.to_numeric(out.get(sdss_i0, np.nan), errors="coerce")

    # Only fill where missing unless overwrite_existing=True
    need_color = out["gi_primary"].isna() if not overwrite_existing else pd.Series(True, index=out.index)

    mask = ok & need_color
    if mask.any():
        out.loc[mask, "gr_primary"] = g0[mask] - r0[mask]
        out.loc[mask, "gi_primary"] = g0[mask] - i0[mask]
        out.loc[mask, "color_src"] = "SDSS(cModel extcorr)"

    # --- Optional sanity hints ---
    # If mV_src is SDSS but Vmag_sdss is missing, that's inconsistent
    if "Vmag_sdss" in out.columns:
        bad = out["mV_src"].eq("SDSS") & out["Vmag_sdss"].isna()
        if bad.any():
            out.loc[bad, "mV_src"] = "SDSS(?)"
            # leave the value, just flag the label

    return out

In [68]:
gals_df = finalize_photometry(gals_df, overwrite_existing=True)


In [81]:
from IPython.display import display

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)

display(gals_df)


,name,ra,dec,otype,z,Vmag_simbad_raw,Bmag_simbad_raw,BV_simbad_raw,gmag_simbad_raw,rmag_simbad_raw,imag_simbad_raw,zmag_simbad_raw,gr_simbad_raw,gi_simbad_raw,major_axis_arcsec,galdim_minaxis,galdim_angle,dim_source,galdim_qual,galdim_bibcode,dim_ref,MV_simbad_raw,show_label,Re_arcsec,otype_family,morph_type,major_kpc,is_large_E,sdss_objid,Re_r_arcsec,Re_r_arcsec_circ,Vmag_sdss,Vmag_sdss_err,V_source,mV_app,mV_src,MV_abs,Re_GCS_scale,Re_GCS_arcsec,n_gcs_prior,n_gcs_red_prior,n_gcs_blue_prior,g_cmodel_sdss,r_cmodel_sdss,i_cmodel_sdss,gerr_cmodel_sdss,rerr_cmodel_sdss,ierr_cmodel_sdss,Ag_sdss,Ar_sdss,Ai_sdss,g0_cmodel_sdss,r0_cmodel_sdss,i0_cmodel_sdss,sdss_phot_ok,gr0_sdss,gi0_sdss,g_model_sdss,r_model_sdss,i_model_sdss,gerr_model_sdss,rerr_model_sdss,ierr_model_sdss,g0_model_sdss,r0_model_sdss,i0_model_sdss,mV_app_col,gi_primary,gr_primary,color_src
0,SDSS J125939.47+275116.5,194.914488,27.854598,G,0.031340,16.944000,17.816999,0.872999,16.851999,16.194000,15.884000,15.786000,0.657999,0.967999,1.300000,NaN,<NA>,NED,,,2007SDSS6.C...0000:,-18.056000,False,0.162500,Regular galaxies,,841.404907,False,1.237667e+18,7.100106,4.281512,16.713860,0.006102,"SDSS(cModelMag g,r)",16.713860,SDSS,-18.286140,3.2,13.700837,1.867370,2.240844,1.493896,17.10436,16.50766,16.24260,0.008724,0.008380,0.009607,0.034769,0.024053,0.017874,17.069591,16.483607,16.224726,True,0.585984,0.844865,16.84402,16.18769,15.88271,0.008332,0.008050,0.009112,16.809251,16.163637,15.864836,Vmag_sdss,0.844865,0.585984,SDSS(cModel extcorr)
1,LEDA 126771,195.015438,27.964525,GiC,0.017780,17.260000,18.490000,1.230000,17.207001,16.400999,16.025999,15.919000,0.806002,1.181002,7.150000,NaN,<NA>,NED,,,2007SDSS6.C...0000:,-17.740000,False,0.893750,Members of systems,NaN,2668.911659,False,1.237667e+18,9.422453,7.926311,17.081481,0.005627,"SDSS(cModelMag g,r)",17.081481,SDSS,-17.918519,3.2,25.364194,1.867370,2.240844,1.493896,17.79433,16.64570,16.29974,0.009407,0.006945,0.008212,0.030749,0.021272,0.015807,17.763581,16.624428,16.283933,True,1.139153,1.479648,17.20481,16.39346,16.02829,0.009123,0.006616,0.007813,17.174061,16.372188,16.012483,Vmag_sdss,1.479648,1.139153,SDSS(cModel extcorr)
2,SDSS J130033.33+275849.3,195.138894,27.980364,AG?,0.017200,17.930000,19.882999,1.952999,19.080999,18.424999,18.046000,17.767000,0.656000,1.035000,5.600000,NaN,<NA>,NED,,,2007SDSS6.C...0000:,-17.070000,False,0.700000,Other,,2023.575699,False,1.237667e+18,2.985155,2.879151,18.451897,0.013294,"SDSS(cModelMag g,r)",18.451897,SDSS,-16.548103,3.2,9.213285,1.867370,2.240844,1.493896,18.88300,18.21784,17.98378,0.019511,0.017997,0.020362,0.035028,0.024232,0.018007,18.847972,18.193608,17.965773,True,0.654364,0.882199,19.07247,18.41958,18.04261,0.018818,0.017248,0.019716,19.037442,18.395348,18.024603,Vmag_sdss,0.882199,0.654364,SDSS(cModel extcorr)
3,2MASX J12592536+2756038,194.855579,27.934492,GiC,0.025310,17.020000,18.094000,1.073999,17.577999,16.871000,16.549999,16.316000,0.706999,1.028000,8.580000,0.143000,90,SIMBAD,C,2006AJ....131.1163S,<NA>,-17.980000,False,1.072500,Members of systems,E/S0,4517.585881,False,1.237667e+18,2.307695,2.193042,17.127525,0.003992,"SDSS(cModelMag g,r)",17.127525,SDSS,-17.872475,3.0,6.579125,1.880278,2.256333,1.504222,17.54884,16.90173,16.59250,0.006365,0.005119,0.005766,0.036081,0.024960,0.018548,17.512759,16.876770,16.573952,True,0.635990,0.938808,17.57126,16.86209,16.54906,0.006326,0.005088,0.005735,17.535179,16.837130,16.530512,Vmag_sdss,0.938808,0.635990,SDSS(cModel extcorr)
4,GMP 2550,195.162417,28.009806,G,0.019787,18.450001,19.149000,0.698999,18.840000,NaN,NaN,17.139999,NaN,NaN,5.640000,NaN,<NA>,NED,,,2007SDSS6.C...0000:,-16.549999,False,0.705000,Regular galaxies,Sa,2337.200564,False,1.237667e+18,0.030223,0.019484,24.891213,0.684727,"SDSS(cModelMag g,r)",24.891213,SDSS,-10.108787,3.0,0.058452,1.880278,2.256333,1.504222,25.11438,24.80202,24.36180,0.967193,0.946121,1.069930,0.035293,0.024415,0.018143,25.079087,24.777605,24.343657,True,0.301483,0.735430,25.11438,24.80202,

In [92]:
# reset to more manageable levels
pd.set_option("display.max_rows", 20)
pd.set_option("display.max_columns", 30)


In [80]:
gals_df["mV_app"] = np.nan
gals_df["mV_src"] = pd.NA

# Priority order (SDSS... Simbad... NED)
use_sdss = gals_df["Vmag_sdss"].notna()
use_simbad = gals_df["Vmag_simbad_raw"].notna() & ~use_sdss

gals_df.loc[use_sdss, "mV_app"] = gals_df.loc[use_sdss, "Vmag_sdss"]
gals_df.loc[use_sdss, "mV_src"] = "SDSS"

gals_df.loc[use_simbad, "mV_app"] = gals_df.loc[use_simbad, "Vmag_simbad_raw"]
gals_df.loc[use_simbad, "mV_src"] = "SIMBAD"

In [72]:
gals_df['MV_abs'] = gals_df['mV_app'] - 35.0

In [73]:
print(gals_df["mV_app"].describe())
print(gals_df["MV_abs"].describe())

count    173.000000
mean      16.833929
std        1.869727
min       11.899091
25%       15.447296
50%       16.996206
75%       18.047103
max       24.891213
Name: mV_app, dtype: float64
count    173.000000
mean     -18.166071
std        1.869727
min      -23.100909
25%      -19.552704
50%      -18.003794
75%      -16.952897
max      -10.108787
Name: MV_abs, dtype: float64


In [74]:

is_luminous = gals_df["MV_abs"] <= -21.5
is_huge     = gals_df["Re_r_arcsec_circ"] >= 20 


In [75]:
len(gals_df[is_luminous])

3

In [76]:
len(gals_df[is_huge])

3

In [77]:
gals_df["is_large_E"] = (is_E & (is_luminous | is_huge)) | is_BCG | has_cD


In [93]:
gals_df[is_E]


,name,ra,dec,otype,z,Vmag_simbad_raw,Bmag_simbad_raw,BV_simbad_raw,gmag_simbad_raw,rmag_simbad_raw,imag_simbad_raw,zmag_simbad_raw,gr_simbad_raw,gi_simbad_raw,major_axis_arcsec,...,gr0_sdss,gi0_sdss,g_model_sdss,r_model_sdss,i_model_sdss,gerr_model_sdss,rerr_model_sdss,ierr_model_sdss,g0_model_sdss,r0_model_sdss,i0_model_sdss,mV_app_col,gi_primary,gr_primary,color_src
0,SDSS J125939.47+275116.5,194.914488,27.854598,G,0.03134,16.944000,17.816999,0.872999,16.851999,16.194000,15.884000,15.786000,0.657999,0.967999,1.300000,...,0.585984,0.844865,16.84402,16.18769,15.88271,0.008332,0.008050,0.009112,16.809251,16.163637,15.864836,Vmag_sdss,0.844865,0.585984,SDSS(cModel extcorr)
1,LEDA 126771,195.015438,27.964525,GiC,0.01778,17.260000,18.490000,1.230000,17.207001,16.400999,16.025999,15.919000,0.806002,1.181002,7.150000,...,1.139153,1.479648,17.20481,16.39346,16.02829,0.009123,0.006616,0.007813,17.174061,16.372188,16.012483,Vmag_sdss,1.479648,1.139153,SDSS(cModel extcorr)
2,SDSS J130033.33+275849.3,195.138894,27.980364,AG?,0.01720,17.930000,19.882999,1.952999,19.080999,18.424999,18.046000,17.767000,0.656000,1.035000,5.600000,...,0.654364,0.882199,19.07247,18.41958,18.04261,0.018818,0.017248,0.019716,19.037442,18.395348,18.024603,Vmag_sdss,0.882199,0.654364,SDSS(cModel extcorr)
5,SDSS J125942.36+280158.5,194.926540,28.032920,G,0.02630,18.760000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.010000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,<NA>
7,SDSS J125926.45+275124.7,194.860237,27.856882,AG?,0.01660,17.719999,19.120001,1.400002,18.222000,17.548000,17.219000,17.108999,0.674000,1.003000,5.700000,...,0.606935,0.902986,18.21467,17.54108,17.21710,0.011330,0.009242,0.011051,18.177537,17.515391,17.198010,Vmag_sdss,0.902986,0.606935,SDSS(cModel extcorr)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
164,LEDA 126781,194.945554,27.991910,G,0.02806,17.430000,18.620001,1.190001,18.250999,17.563000,17.235001,17.037001,0.688000,1.015999,7.310000,...,0.670030,0.969831,18.25128,17.56304,17.23544,0.009389,0.008103,0.009230,18.224446,17.544476,17.221645,Vmag_sdss,0.969831,0.670030,SDSS(cModel extcorr)
165,LEDA 44809,195.178493,27.963119,EmG,0.02809,14.960000,16.013000,1.053000,15.470000,14.708000,14.333000,14.034000,0.762000,1.137000,38.448418,...,0.666377,1.044507,15.46956,14.70831,14.33325,0.002737,0.002342,0.002454,15.435515,14.684758,14.315748,Vmag_sdss,1.044507,0.666377,SDSS(cModel extcorr)
167,SDSS J130041.19+280242.4,195.171675,28.045127,G,0.02877,17.150000,18.487000,1.337000,18.054001,17.327999,16.976999,16.693001,0.726002,1.077002,7.070000,...,0.726279,1.077916,18.05374,17.32846,16.97660,0.006821,0.005682,0.006168,18.018047,17.303768,16.958251,Vmag_sdss,1.077916,0.726279,SDSS(cModel extcorr)
168,LEDA 1821892,195.143442,27.934682,G,0.02923,16.809999,18.511999,1.702000,17.386000,16.688999,16.424000,16.160000,0.697001,0.962000,12.599999,...,0.769137,0.890724,17.38590,16.68892,16.42357,0.010066,0.008036,0.012428,17.352405,16.665749,16.406351,Vmag_sdss,0.890724,0.769137,SDSS(cModel extcorr)


In [94]:
gals_df.sort_values(by='Re_r_arcsec_circ', ascending=False).head(30)


,name,ra,dec,otype,z,Vmag_simbad_raw,Bmag_simbad_raw,BV_simbad_raw,gmag_simbad_raw,rmag_simbad_raw,imag_simbad_raw,zmag_simbad_raw,gr_simbad_raw,gi_simbad_raw,major_axis_arcsec,...,gr0_sdss,gi0_sdss,g_model_sdss,r_model_sdss,i_model_sdss,gerr_model_sdss,rerr_model_sdss,ierr_model_sdss,g0_model_sdss,r0_model_sdss,i0_model_sdss,mV_app_col,gi_primary,gr_primary,color_src
106,NGC 4874,194.898789,27.959248,LIN,0.02391,12.710000,13.700000,0.990000,12.647200,11.797900,11.660,11.1535,0.849299,0.987200,140.404205,...,0.795435,1.254257,12.79304,11.98041,11.52575,0.001763,0.001691,0.001796,12.762947,11.959592,11.510280,Vmag_sdss,1.254257,0.795435,SDSS(cModel extcorr)
96,NGC 4889,195.033738,27.977025,EmG,0.02150,11.300000,13.000000,1.700000,12.352800,11.501700,11.527,10.8346,0.851100,0.825800,175.404602,...,0.817721,1.033334,12.34061,11.51430,11.21793,0.001639,0.001641,0.001738,12.308947,11.492396,11.201653,Vmag_sdss,1.033334,0.817721,SDSS(cModel extcorr)
31,LEDA 44708,195.025448,27.978315,GiC,0.02546,15.994000,16.900000,0.905999,15.673000,15.063000,14.668,14.4700,0.610001,1.005000,33.000000,...,0.723008,0.685847,14.86612,14.16533,13.99254,0.003643,0.002911,0.003584,14.834997,14.143799,13.976540,Vmag_sdss,0.685847,0.723008,SDSS(cModel extcorr)
125,SDSS J130051.15+280249.7,195.213150,28.047139,GiC,0.02116,17.077000,17.804001,0.727001,16.540001,15.845000,15.509,15.3380,0.695001,1.031001,15.450000,...,0.768613,1.045245,16.54006,15.84471,15.50868,0.007015,0.005982,0.006731,16.504962,15.820429,15.490637,Vmag_sdss,1.045245,0.768613,SDSS(cModel extcorr)
114,IC 4051,195.226929,28.007639,EmG,0.01662,13.500000,14.800000,1.300000,13.830000,13.045000,12.617,12.2890,0.785000,1.213000,79.127998,...,0.680840,1.110067,13.83039,13.04475,12.61681,0.001806,0.001685,0.001694,13.797067,13.021697,12.599679,Vmag_sdss,1.110067,0.680840,SDSS(cModel extcorr)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,NGC 4873,194.886649,27.983618,EmG,0.01943,14.360000,15.400000,1.040000,14.940000,14.138000,13.737,13.4440,0.802000,1.202999,24.180000,...,0.726462,1.200471,14.83407,14.04072,13.63580,0.002264,0.002025,0.002056,14.805199,14.020747,13.620958,Vmag_sdss,1.200471,0.726462,SDSS(cModel extcorr)
162,NGC 4883,194.983405,28.034716,GiG,0.02719,14.270000,15.200000,0.929999,14.821000,14.007000,13.584,13.2540,0.814000,1.237000,38.738400,...,0.790702,1.225029,14.82103,14.00690,13.58407,0.002165,0.001958,0.001964,14.791023,13.986142,13.568644,Vmag_sdss,1.225029,0.790702,SDSS(cModel extcorr)
159,LEDA 93696,194.769930,28.050391,G,0.02689,16.530001,18.059999,1.529999,17.004999,16.290001,15.933,15.6540,0.714998,1.072000,10.980000,...,0.694661,1.045960,17.00517,16.29045,15.93348,0.005736,0.004853,0.005271,16.972304,16.267714,15.916584,Vmag_sdss,1.045960,0.694661,SDSS(cModel extcorr)
98,IC 3955,194.775123,27.996709,GiC,0.02557,14.310000,15.591000,1.280999,15.059000,14.227300,13.806,13.5935,0.831700,1.253000,43.197418,...,0.793913,1.240104,14.98426,14.16541,13.76454,0.002362,0.002058,0.002109,14.949488,14.141355,13.746665,Vmag_sdss,1.240104,0.793913,SDSS(cModel extcorr)


In [84]:
def attach_gc_Re_defaults(gals_df):
    df = gals_df.copy()
    # pick galaxy Re (prefer circularized)
    Re_gal_arcsec = df.get("Re_r_arcsec_circ").fillna(df.get("Re_r_arcsec"))
    # scaling factor by class
    scale = np.where(df.get("is_large_E", False), 4.8, np.where(is_E, 3.2, 3.0))
    df["Re_GCS_scale"] = scale
    df["Re_GCS_arcsec"] = Re_gal_arcsec.astype(float) * scale

    return df

gals_df = attach_gc_Re_defaults(gals_df)


In [95]:
gals_df.sort_values(by='Re_r_arcsec_circ', ascending=False).head(30)


,name,ra,dec,otype,z,Vmag_simbad_raw,Bmag_simbad_raw,BV_simbad_raw,gmag_simbad_raw,rmag_simbad_raw,imag_simbad_raw,zmag_simbad_raw,gr_simbad_raw,gi_simbad_raw,major_axis_arcsec,...,gr0_sdss,gi0_sdss,g_model_sdss,r_model_sdss,i_model_sdss,gerr_model_sdss,rerr_model_sdss,ierr_model_sdss,g0_model_sdss,r0_model_sdss,i0_model_sdss,mV_app_col,gi_primary,gr_primary,color_src
106,NGC 4874,194.898789,27.959248,LIN,0.02391,12.710000,13.700000,0.990000,12.647200,11.797900,11.660,11.1535,0.849299,0.987200,140.404205,...,0.795435,1.254257,12.79304,11.98041,11.52575,0.001763,0.001691,0.001796,12.762947,11.959592,11.510280,Vmag_sdss,1.254257,0.795435,SDSS(cModel extcorr)
96,NGC 4889,195.033738,27.977025,EmG,0.02150,11.300000,13.000000,1.700000,12.352800,11.501700,11.527,10.8346,0.851100,0.825800,175.404602,...,0.817721,1.033334,12.34061,11.51430,11.21793,0.001639,0.001641,0.001738,12.308947,11.492396,11.201653,Vmag_sdss,1.033334,0.817721,SDSS(cModel extcorr)
31,LEDA 44708,195.025448,27.978315,GiC,0.02546,15.994000,16.900000,0.905999,15.673000,15.063000,14.668,14.4700,0.610001,1.005000,33.000000,...,0.723008,0.685847,14.86612,14.16533,13.99254,0.003643,0.002911,0.003584,14.834997,14.143799,13.976540,Vmag_sdss,0.685847,0.723008,SDSS(cModel extcorr)
125,SDSS J130051.15+280249.7,195.213150,28.047139,GiC,0.02116,17.077000,17.804001,0.727001,16.540001,15.845000,15.509,15.3380,0.695001,1.031001,15.450000,...,0.768613,1.045245,16.54006,15.84471,15.50868,0.007015,0.005982,0.006731,16.504962,15.820429,15.490637,Vmag_sdss,1.045245,0.768613,SDSS(cModel extcorr)
114,IC 4051,195.226929,28.007639,EmG,0.01662,13.500000,14.800000,1.300000,13.830000,13.045000,12.617,12.2890,0.785000,1.213000,79.127998,...,0.680840,1.110067,13.83039,13.04475,12.61681,0.001806,0.001685,0.001694,13.797067,13.021697,12.599679,Vmag_sdss,1.110067,0.680840,SDSS(cModel extcorr)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,NGC 4873,194.886649,27.983618,EmG,0.01943,14.360000,15.400000,1.040000,14.940000,14.138000,13.737,13.4440,0.802000,1.202999,24.180000,...,0.726462,1.200471,14.83407,14.04072,13.63580,0.002264,0.002025,0.002056,14.805199,14.020747,13.620958,Vmag_sdss,1.200471,0.726462,SDSS(cModel extcorr)
162,NGC 4883,194.983405,28.034716,GiG,0.02719,14.270000,15.200000,0.929999,14.821000,14.007000,13.584,13.2540,0.814000,1.237000,38.738400,...,0.790702,1.225029,14.82103,14.00690,13.58407,0.002165,0.001958,0.001964,14.791023,13.986142,13.568644,Vmag_sdss,1.225029,0.790702,SDSS(cModel extcorr)
159,LEDA 93696,194.769930,28.050391,G,0.02689,16.530001,18.059999,1.529999,17.004999,16.290001,15.933,15.6540,0.714998,1.072000,10.980000,...,0.694661,1.045960,17.00517,16.29045,15.93348,0.005736,0.004853,0.005271,16.972304,16.267714,15.916584,Vmag_sdss,1.045960,0.694661,SDSS(cModel extcorr)
98,IC 3955,194.775123,27.996709,GiC,0.02557,14.310000,15.591000,1.280999,15.059000,14.227300,13.806,13.5935,0.831700,1.253000,43.197418,...,0.793913,1.240104,14.98426,14.16541,13.76454,0.002362,0.002058,0.002109,14.949488,14.141355,13.746665,Vmag_sdss,1.240104,0.793913,SDSS(cModel extcorr)


In [86]:
import numpy as np

def sersic_n_priors(df, re_gcs_col="Re_GCS_arcsec",
                    re_gal_cols=("Re_r_arcsec_circ","Re_r_arcsec"),
                    is_bcg_col="is_bcg", mstar_col="Mstar"):
    out = df.copy()
    Re_gcs = out[re_gcs_col].astype(float)
    # pick a galaxy Re
    Re_gal = None
    for c in re_gal_cols:
        if c in out.columns:
            Re_gal = out[c]
            if Re_gal.notna().any(): break
    Re_gal = pd.to_numeric(Re_gal, errors="coerce")
    eta = Re_gcs / Re_gal

    # base n_total from eta (clip to safe range)
    n_total = 2.1 - 0.2*np.log(eta.replace([np.inf, -np.inf], np.nan))
    n_total = np.clip(n_total, 0.7, 3.0)

    # BCG/cD tweak: shallower by 0.2
    is_bcg = np.zeros(len(out), dtype=bool)
    if is_bcg_col in out.columns:
        is_bcg |= out[is_bcg_col].fillna(False).astype(bool).values
    if mstar_col in out.columns:
        is_bcg |= (pd.to_numeric(out[mstar_col], errors="coerce") >= 3e11).fillna(False).values
    n_total = np.where(is_bcg, n_total - 0.2, n_total)

    out["n_gcs_prior"]       = n_total
    out["n_gcs_red_prior"]   = np.clip(1.2*n_total, 0.8, 3.5)
    out["n_gcs_blue_prior"]  = np.clip(0.8*n_total, 0.6, 2.8)

    return out


In [87]:
gals_df = sersic_n_priors(gals_df)


In [96]:
gals_df.sort_values(by='Re_r_arcsec_circ', ascending=False).head(30)


,name,ra,dec,otype,z,Vmag_simbad_raw,Bmag_simbad_raw,BV_simbad_raw,gmag_simbad_raw,rmag_simbad_raw,imag_simbad_raw,zmag_simbad_raw,gr_simbad_raw,gi_simbad_raw,major_axis_arcsec,...,gr0_sdss,gi0_sdss,g_model_sdss,r_model_sdss,i_model_sdss,gerr_model_sdss,rerr_model_sdss,ierr_model_sdss,g0_model_sdss,r0_model_sdss,i0_model_sdss,mV_app_col,gi_primary,gr_primary,color_src
106,NGC 4874,194.898789,27.959248,LIN,0.02391,12.710000,13.700000,0.990000,12.647200,11.797900,11.660,11.1535,0.849299,0.987200,140.404205,...,0.795435,1.254257,12.79304,11.98041,11.52575,0.001763,0.001691,0.001796,12.762947,11.959592,11.510280,Vmag_sdss,1.254257,0.795435,SDSS(cModel extcorr)
96,NGC 4889,195.033738,27.977025,EmG,0.02150,11.300000,13.000000,1.700000,12.352800,11.501700,11.527,10.8346,0.851100,0.825800,175.404602,...,0.817721,1.033334,12.34061,11.51430,11.21793,0.001639,0.001641,0.001738,12.308947,11.492396,11.201653,Vmag_sdss,1.033334,0.817721,SDSS(cModel extcorr)
31,LEDA 44708,195.025448,27.978315,GiC,0.02546,15.994000,16.900000,0.905999,15.673000,15.063000,14.668,14.4700,0.610001,1.005000,33.000000,...,0.723008,0.685847,14.86612,14.16533,13.99254,0.003643,0.002911,0.003584,14.834997,14.143799,13.976540,Vmag_sdss,0.685847,0.723008,SDSS(cModel extcorr)
125,SDSS J130051.15+280249.7,195.213150,28.047139,GiC,0.02116,17.077000,17.804001,0.727001,16.540001,15.845000,15.509,15.3380,0.695001,1.031001,15.450000,...,0.768613,1.045245,16.54006,15.84471,15.50868,0.007015,0.005982,0.006731,16.504962,15.820429,15.490637,Vmag_sdss,1.045245,0.768613,SDSS(cModel extcorr)
114,IC 4051,195.226929,28.007639,EmG,0.01662,13.500000,14.800000,1.300000,13.830000,13.045000,12.617,12.2890,0.785000,1.213000,79.127998,...,0.680840,1.110067,13.83039,13.04475,12.61681,0.001806,0.001685,0.001694,13.797067,13.021697,12.599679,Vmag_sdss,1.110067,0.680840,SDSS(cModel extcorr)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,NGC 4873,194.886649,27.983618,EmG,0.01943,14.360000,15.400000,1.040000,14.940000,14.138000,13.737,13.4440,0.802000,1.202999,24.180000,...,0.726462,1.200471,14.83407,14.04072,13.63580,0.002264,0.002025,0.002056,14.805199,14.020747,13.620958,Vmag_sdss,1.200471,0.726462,SDSS(cModel extcorr)
162,NGC 4883,194.983405,28.034716,GiG,0.02719,14.270000,15.200000,0.929999,14.821000,14.007000,13.584,13.2540,0.814000,1.237000,38.738400,...,0.790702,1.225029,14.82103,14.00690,13.58407,0.002165,0.001958,0.001964,14.791023,13.986142,13.568644,Vmag_sdss,1.225029,0.790702,SDSS(cModel extcorr)
159,LEDA 93696,194.769930,28.050391,G,0.02689,16.530001,18.059999,1.529999,17.004999,16.290001,15.933,15.6540,0.714998,1.072000,10.980000,...,0.694661,1.045960,17.00517,16.29045,15.93348,0.005736,0.004853,0.005271,16.972304,16.267714,15.916584,Vmag_sdss,1.045960,0.694661,SDSS(cModel extcorr)
98,IC 3955,194.775123,27.996709,GiC,0.02557,14.310000,15.591000,1.280999,15.059000,14.227300,13.806,13.5935,0.831700,1.253000,43.197418,...,0.793913,1.240104,14.98426,14.16541,13.76454,0.002362,0.002058,0.002109,14.949488,14.141355,13.746665,Vmag_sdss,1.240104,0.793913,SDSS(cModel extcorr)


In [97]:
logger.debug("Checking for sparsity in color data")
gals_df[['gi_primary', 'gr_primary', 'BV_simbad_raw']].isna().sum()


[DEBUG] 19:09:29.597 Checking for sparsity in color data


gi_primary       12
gr_primary       12
BV_simbad_raw    17
dtype: int64

In [98]:
logger.debug("Setting (g-i) as default galaxy color (if available).")
gals_df['gal_color'] = gals_df['gi_primary']
gals_df['gal_color_src'] = 'g-i'

logger.debug("Fallback to (B-V) galaxy color (if available).")
mask = gals_df['gal_color'].isna()
gals_df.loc[mask, 'gal_color'] = gals_df.loc[mask, 'BV_simbad_raw']
gals_df.loc[mask, 'gal_color_src'] = 'B-V'

logger.debug("Final fallback to (g-r) galaxy color (if available).")
mask = gals_df['gal_color'].isna()
gals_df.loc[mask, 'gal_color'] = gals_df.loc[mask, 'gr_primary']
gals_df.loc[mask, 'gal_color_src'] = 'g-r'


[DEBUG] 19:14:19.100 Setting (g-i) as default galaxy color (if available).
[DEBUG] 19:14:19.102 Fallback to (B-V) galaxy color (if available).
[DEBUG] 19:14:19.105 Final fallback to (g-r) galaxy color (if available).


In [99]:
logger.debug("Checking for final color data")
gals_df[['gal_color']].isna().sum()

[DEBUG] 19:14:29.123 Checking for final color data


gal_color    3
dtype: int64

In [102]:
df_missing = gals_df[gals_df['gal_color'].isna()]
df_missing[['name', 'mV_app', 'MV_abs', 'z', 'otype']]


,name,mV_app,MV_abs,z,otype
5,SDSS J125942.36+280158.5,18.760000,-16.240000,0.026300,G
40,CAIRNS J125947.21+280315.7,18.030001,-16.969999,0.031799,GiC
173,Coma cluster,NaN,NaN,0.021500,CLUSTER


In [101]:
gals_df.columns

Index(['name', 'ra', 'dec', 'otype', 'z', 'Vmag_simbad_raw', 'Bmag_simbad_raw',
       'BV_simbad_raw', 'gmag_simbad_raw', 'rmag_simbad_raw',
       'imag_simbad_raw', 'zmag_simbad_raw', 'gr_simbad_raw', 'gi_simbad_raw',
       'major_axis_arcsec', 'galdim_minaxis', 'galdim_angle', 'dim_source',
       'galdim_qual', 'galdim_bibcode', 'dim_ref', 'MV_simbad_raw',
       'show_label', 'Re_arcsec', 'otype_family', 'morph_type', 'major_kpc',
       'is_large_E', 'sdss_objid', 'Re_r_arcsec', 'Re_r_arcsec_circ',
       'Vmag_sdss', 'Vmag_sdss_err', 'V_source', 'mV_app', 'mV_src', 'MV_abs',
       'Re_GCS_scale', 'Re_GCS_arcsec', 'n_gcs_prior', 'n_gcs_red_prior',
       'n_gcs_blue_prior', 'g_cmodel_sdss', 'r_cmodel_sdss', 'i_cmodel_sdss',
       'gerr_cmodel_sdss', 'rerr_cmodel_sdss', 'ierr_cmodel_sdss', 'Ag_sdss',
       'Ar_sdss', 'Ai_sdss', 'g0_cmodel_sdss', 'r0_cmodel_sdss',
       'i0_cmodel_sdss', 'sdss_phot_ok', 'gr0_sdss', 'gi0_sdss',
       'g_model_sdss', 'r_model_sdss', 'i_model_s

In [111]:
logger.debug("Saving GALAXY ARCHIVE DATAFRAME to csv...")
gals_df.to_csv('data/gals_data_from_archives.csv', index=False)


[DEBUG] 19:21:11.888 Saving GALAXY ARCHIVE DATAFRAME to csv...


In [104]:
gals_df.loc[gals_df['name'] == 'Coma cluster'].iloc[0]


name             Coma cluster
ra                 195.017071
dec                 27.977025
otype                 CLUSTER
z                      0.0215
                     ...     
gi_primary                NaN
gr_primary                NaN
color_src                <NA>
gal_color                 NaN
gal_color_src             g-r
Name: 173, Length: 72, dtype: object

In [109]:
gals_df.loc[gals_df['name'] == 'NGC 4889'].iloc[0]


name                         NGC 4889
ra                         195.033738
dec                         27.977025
otype                             EmG
z                              0.0215
                         ...         
gi_primary                   1.033334
gr_primary                   0.817721
color_src        SDSS(cModel extcorr)
gal_color                    1.033334
gal_color_src                     g-i
Name: 96, Length: 72, dtype: object

In [110]:
gals_df[gals_df['name']=='NGC 4921']['show_label']

Series([], Name: show_label, dtype: boolean)

In [ ]:
gals_df[gals_df['Re_arcsec']<=2.9]


In [ ]:
gals_df[gals_df['Re_arcsec']<=2.9]


In [ ]:
#inFileName = "allpointings_master.dat_NoOverlap"
# inFileName = "data/newallpointingsmaster_26may2021.dat"  # with delimeter ' '
inFileName = "data/inspectedpointings_merged_19Oct2024.csv"  # with delimieter ','

inFile = open(inFileName, 'r')
lines = inFile.readlines()
inFile.close

data = pd.read_csv(inFileName, delimiter=',')


and then we need to filter for color on CSS in general, and additional filter for UCDs on the tighter color range and luminosity, and outlined in paper 1. Color value are 

- CSS: *color* 0.5 < (F475W − F814W ) < 2.5 and 
- UCD: *color* 1.3 < (F475W − F814W ) < 2.1 and *magnitude* F814W < 22.9 mag


In [ ]:
data = data[(data['color'] > 0.5) & (data['color'] < 2.5)]

# Define the range for UCD 'color' and 'mag_814'
ucd_color_min, ucd_color_max = 1.3, 2.1     #
ucd_mag_814_min, ucd_mag_814_max = 0, 22.9  # should be 0 mag to 22.9 mag

# remove duplicated data
data = data.drop_duplicates()

# Filter the DataFrame
gcs_df = data[(data['color'] <= ucd_color_min) | (data['color'] >= ucd_color_max) |
                 (data['mag_814'] <= ucd_mag_814_min) | (data['mag_814'] >= ucd_mag_814_max)]
ucd_df = data[(data['color'] > ucd_color_min) & (data['color'] < ucd_color_max) &
                 (data['mag_814'] > ucd_mag_814_min) & (data['mag_814'] < ucd_mag_814_max)]


In [ ]:
total_gcs = len(gcs_df)
total_ucds = len(ucd_df)
total_css = len(data)

print('Total GCs: ', total_gcs)
print('Total UCDs: ', total_ucds)
print('Total count: ', total_css)
print('Mag F814W min, max:', np.min(data['mag_814']), np.max(data['mag_814']) )
print('Mag F475W min, max:', np.min(data['mag_475']), np.max(data['mag_475']) )

data.head(5)

In [ ]:
data.columns

In [ ]:
gals_df.head(5)


In [ ]:
def get_sat_coords_and_Re(gals_df, sat_list):

    # if the list of satellites is empty
    if not sat_list:
        return None 
    
    # Filter rows in gals_df
    sat_df = gals_df[gals_df['name'].isin(sat_list)]

    # Create SkyCoord list
    sat_name = sat_df['name'].values
    sat_coords = SkyCoord(ra=sat_df['ra'].values * u.deg,
                          dec=sat_df['dec'].values * u.deg)

    # Extract Re values
    sat_re_arcsec = sat_df['Re_arcsec'].values  # NumPy array

    # extract label display boolean
    sat_show_label = sat_df['show_label'].values  # NumPy array

    # (Optional) zip into pairs
    sat_coords_re = list(zip(sat_name, sat_coords, sat_re_arcsec, sat_show_label))
    
    return sat_coords_re

def get_sat_seps(center, sat_coords_re):
    
    # if the list of satellites is empty
    if not sat_coords_re:
        return None
    
    # unzip the sat list
    sat_names, sat_coords, sat_re_arcsec, show = zip(*sat_coords_re)
    sat_coords = SkyCoord(sat_coords)

    # determine the radial separation of the satellites from the center (galaxy)
    sat_sep_arcsec = center.separation(sat_coords).arcsec  # array of separations

    # zip back into a list
    sat_coords_re_sep = list(zip(sat_names, sat_coords, sat_re_arcsec, sat_sep_arcsec, show))

    return sat_coords_re_sep

sat_coords_re = get_sat_coords_and_Re(gals_df, ['NGC 4871', 'NGC 4872', 'NGC 4873', 'IC 3998'])
sat_coords_re



In [ ]:
print(gals_df[gals_df['Re_arcsec'] != 0])


In [ ]:
tmp_df = gals_df[gals_df['Re_arcsec']!=gals_df['Re_arcsec']].sort_values(by='MajAxis (arcmin)')
tmp_df


In [ ]:
tmp_df[tmp_df['name']=='NGC 4889']


In [ ]:
[name for name in gals_df[gals_df['name']!='NGC 4889']['name']]


## 